# Notebook 2: Data Curation and Dataset Characterisation — GPCR Benchmark Panel

**Purpose:** Apply a 10-step curation pipeline to raw ChEMBL bioactivity data
(structure standardisation, deduplication, PAINS/Brenk annotation, descriptor
annotation) AND generate full publication-ready dataset-characterisation
reporting (tables + figures) on the result — this notebook covers both, not
just cleaning. Loops over **all five targets (DRD2, CB2, ADORA2A,
OPRM1, CCR5) AND all three activity pools** (`ki`/`ki_ic50`/`full`) in a
single run — cleaning is cheap (no training/tuning), so there's no reason to
rely on someone remembering to manually re-run this notebook per target/pool.
Structure standardisation (Step 1, the slow RDKit part) runs once per target
and is reused across all three pools; only the pool-dependent steps (2
onward) actually run 15 times (5 targets x 3 pools).

Running all five targets through the identical curation functions (same
thresholds, executed once, not duplicated/copy-pasted per target) is what
makes cross-target comparison valid — this is the methodology-consistency
requirement the whole benchmark depends on.

**Activity harmonisation:** Ki, IC50, and EC50 measurements are independently
curated, standardised to their negative logarithmic values (pKi, pIC50, pEC50
— all on the same -log10(M) scale), and combined into a unified potency
dataset following rigorous quality control. All three pools (`ki` / `ki_ic50`
/ `full`) are produced every run for every target, supporting a three-way
sensitivity analysis of activity-type harmonisation on predictive performance.

**No cross-target selectivity pairing in this notebook.** An earlier two-target design paired targets for selectivity analysis. This panel's
five targets are pharmacologically independent, not a selectivity pair, so
that concept doesn't carry over. What this notebook produces instead is
**per-target** dataset characterisation (scaffold diversity, class balance,
physicochemical space, structural alerts, etc.) for all 5 targets x 3 pools —
per-target inputs, not a cross-target comparison. The actual cross-target
comparison tables/figures (the meta-analysis layer) belong in a dedicated
later notebook that reads these per-target outputs.

**Required inputs:** `data/raw/raw_data_{drd2,cb2,adora2a,oprm1,ccr5}.csv` (from notebook 01)

**Generated outputs (all in `data/processed/`, full list + SHA-256 hashes in
`manifest_02_data_cleaning.json`):**
- 15 cleaned datasets (`cleaned_data_{target}_{ki,ki_ic50,full}.csv`) + 5 `step1_clean_data_{target}.csv`
- Master dataset summary (`dataset_summary_all_targets_pools.csv`) and supporting tables:
  cleaning attrition, endpoint composition, structural alerts (summary + frequency),
  assay/measurement support, physicochemical descriptors, scaffold diversity
  (summary + per-compound assignments + frequency table), activity-pool increment
  analysis, cross-pool overlap/nesting check
- 10 publication-ready figures (`figures/`, 300dpi PNG + vector PDF) with plot-source
  data saved alongside every one — dataset attrition, pool expansion, class balance,
  pActivity distributions, endpoint composition, scaffold diversity, property
  distributions, chemical-space PCA, assay support, structural alerts


In [1]:
! pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 44.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.rdBase import BlockLogs
from rdkit.Chem import FilterCatalog
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict

import warnings
from tqdm import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')


In [5]:
from pathlib import Path

HPC_MODE = False  # Set True when running on HPC

if HPC_MODE:
    PROJECT_DIR = Path("./")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/My Drive/gpcr_benchmark")

for subdir in ["data/raw", "data/processed", "data/processed/figures",
               "data/external/drugbank", "ml/results", "ml/models"]:
    (PROJECT_DIR / subdir).mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/My Drive/gpcr_benchmark


In [6]:
base_path      = PROJECT_DIR / "data" / "raw"
processed_path = PROJECT_DIR / "data" / "processed"

In [7]:
# =============================================================================
# USER CONFIGURATION
# =============================================================================

# All three activity-type pools are cleaned in this single run (see notebook
# header) — no ACTIVITY_POOL single-select any more, just POOLS.
POOL_TYPES = {
    'ki':      ['Ki'],
    'ki_ic50': ['Ki', 'IC50'],
    'full':    ['Ki', 'IC50', 'EC50'],
}
POOLS = ['ki', 'ki_ic50', 'full']
assert all(p in POOL_TYPES for p in POOLS), f'POOLS must be a subset of {list(POOL_TYPES)}'

# pActivity threshold for binary classification (active=1, inactive=0).
# pActivity = -log10(value in M).  6.0 = 1 uM.  Same threshold across all pools
# AND all targets — this is the fixed, pre-defined pipeline convention the
# whole benchmark depends on (any target-specific deviation must be justified
# and disclosed explicitly, never silent).
ACTIVITY_THRESHOLD = 6.0

# All 5 targets are cleaned in this single run — cleaning is cheap, unlike
# notebook 03's expensive per-target training/tuning.
TARGETS = ['drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5']

assert 0.0 < ACTIVITY_THRESHOLD < 14.0, f'ACTIVITY_THRESHOLD out of plausible pActivity range: {ACTIVITY_THRESHOLD}'
assert all(tk in ('drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5') for tk in TARGETS), f'Unexpected target in TARGETS: {TARGETS}'

print(f'Pools             : {POOLS}')
print(f'Active threshold  : pActivity >= {ACTIVITY_THRESHOLD}')
print(f'Targets           : {TARGETS}')


Pools             : ['ki', 'ki_ic50', 'full']
Active threshold  : pActivity >= 6.0
Targets           : ['drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5']


In [8]:
# Load raw data for both targets
df_raw = {}
for tk in TARGETS:
    df_raw[tk] = pd.read_csv(base_path / f'raw_data_{tk}.csv')
    print(f'{tk.upper()}: starting with {len(df_raw[tk])} raw records')

DRD2: starting with 17263 raw records
CB2: starting with 13662 raw records
ADORA2A: starting with 10554 raw records
OPRM1: starting with 14009 raw records
CCR5: starting with 3449 raw records


In [9]:
# =============================================================================
# STEP 1: Chemical Structure Cleaning
# =============================================================================

In [10]:
print("\n=== STEP 1: Chemical Structure Cleaning ===")

USE_CACHED_STEP1 = False  # True: load previously-saved step1_clean_data_{target}.csv
                           # instead of re-cleaning. Kept in the SAME cell as the
                           # cleaning code (not a separate cell below it) so a
                           # "Run All" can't clean fresh and then immediately
                           # overwrite the result with a stale cache.

def clean_structures(df):
    """Clean and standardise chemical structures. Returns (df_clean, exclusion_stats)
    — exclusion_stats distinguishes missing-SMILES from failed-standardisation
    (parsing/Cleanup/FragmentParent/Normalizer/Uncharger/Tautomer canonicalisation
    all collapse to one 'failed_standardisation' bucket rather than a full
    per-step reason taxonomy — the coarse 2-way split is what the row-level
    data actually distinguishes without deeper instrumentation of each RDKit
    call, and is enough for an audit trail without turning this into a much
    slower per-step try/except pipeline)."""
    df_clean = df.copy()

    initial_count = len(df_clean)
    df_clean = df_clean.dropna(subset=['canonical_smiles'])
    n_missing_smiles = initial_count - len(df_clean)
    print(f'Removed {n_missing_smiles} rows with missing SMILES')

    valid_mols = []
    clean_smiles = []

    block = BlockLogs()
    normalizer = rdMolStandardize.Normalizer()
    uncharger = rdMolStandardize.Uncharger()
    te = rdMolStandardize.TautomerEnumerator()

    print('Processing and standardising chemical structures...')
    for smiles in tqdm(df_clean['canonical_smiles'], desc='Validating structures'):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is not None:
                mol = rdMolStandardize.Cleanup(mol)
                mol = rdMolStandardize.FragmentParent(mol)
                mol = normalizer.normalize(mol)
                mol = uncharger.uncharge(mol)
                canon_mol = te.Canonicalize(mol)
                canon_smi = Chem.MolToSmiles(canon_mol, canonical=True, isomericSmiles=True)
                valid_mols.append(True)
                clean_smiles.append(canon_smi)
            else:
                valid_mols.append(False)
                clean_smiles.append(None)
        except Exception:
            valid_mols.append(False)
            clean_smiles.append(None)
    del block

    df_clean['valid_structure'] = valid_mols
    df_clean['clean_smiles'] = clean_smiles

    before_validation = len(df_clean)
    df_clean = df_clean[df_clean['valid_structure']].dropna(subset=['clean_smiles'])
    n_failed_standardisation = before_validation - len(df_clean)
    print(f'Removed {n_failed_standardisation} invalid/failed-standardisation structures')

    exclusion_stats = {
        'initial_records': initial_count,
        'missing_smiles_removed': n_missing_smiles,
        'failed_standardisation_removed': n_failed_standardisation,
        'final_valid_structures': len(df_clean),
    }
    return df_clean, exclusion_stats

# Keyed by target ONLY — pool-independent, reused across all 3 pools below
df_step1 = {}
structure_exclusion_records = []
if USE_CACHED_STEP1:
    for tk in TARGETS:
        df_step1[tk] = pd.read_csv(processed_path / f'step1_clean_data_{tk}.csv')
        print(f'Loaded {len(df_step1[tk])} {tk.upper()} records from step1_clean_data_{tk}.csv (cached)')
else:
    for tk in TARGETS:
        print(f'\n>>> {tk.upper()} <<<')
        df_step1[tk], excl_stats = clean_structures(df_raw[tk])
        excl_stats['target'] = tk
        structure_exclusion_records.append(excl_stats)
        df_step1[tk].to_csv(processed_path / f'step1_clean_data_{tk}.csv', index=False)
        print(f'Saved {len(df_step1[tk])} records to step1_clean_data_{tk}.csv')



=== STEP 1: Chemical Structure Cleaning ===

>>> DRD2 <<<
Removed 26 rows with missing SMILES
Processing and standardising chemical structures...


Validating structures: 100%|██████████| 17237/17237 [03:53<00:00, 73.83it/s] 


Removed 0 invalid/failed-standardisation structures
Saved 17237 records to step1_clean_data_drd2.csv

>>> CB2 <<<
Removed 1 rows with missing SMILES
Processing and standardising chemical structures...


Validating structures: 100%|██████████| 13661/13661 [01:22<00:00, 166.44it/s]


Removed 0 invalid/failed-standardisation structures
Saved 13661 records to step1_clean_data_cb2.csv

>>> ADORA2A <<<
Removed 3 rows with missing SMILES
Processing and standardising chemical structures...


...

Removed 0 invalid/failed-standardisation structures
Saved 10551 records to step1_clean_data_adora2a.csv

>>> OPRM1 <<<
Removed 37 rows with missing SMILES
Processing and standardising chemical structures...


Validating structures: 100%|██████████| 13972/13972 [12:50<00:00, 18.14it/s]


Removed 0 invalid/failed-standardisation structures
Saved 13972 records to step1_clean_data_oprm1.csv

>>> CCR5 <<<
Removed 0 rows with missing SMILES
Processing and standardising chemical structures...


Validating structures: 100%|██████████| 3449/3449 [00:53<00:00, 64.94it/s]


Removed 0 invalid/failed-standardisation structures
Saved 3449 records to step1_clean_data_ccr5.csv


In [11]:
# =============================================================================
# STEP 2: Activity Type Selection and pActivity Calculation
# =============================================================================

In [12]:
print("\n=== STEP 2: Activity Type Selection and pActivity Calculation ===")

def convert_and_validate_data(df, allowed_types):
    """
    Filter to the pooled activity types, convert units to nM, and compute
    row-level pActivity = -log10(value in M). Ki, IC50, and EC50 are all
    molar-concentration measures so share the same conversion formula and
    land on the same pKi/pIC50/pEC50 scale, ready to be combined per
    compound in the next step.
    """
    df_conv = df.copy()

    counts = df_conv['standard_type'].value_counts()
    print('Activity types available (raw):')
    print(counts.to_string())

    df_conv = df_conv[df_conv['standard_type'].isin(allowed_types)].copy()
    print(f'\nPooling {allowed_types} -> {len(df_conv)} records '
          f'({len(df_conv) / len(df):.1%} of input)')
    print('Breakdown of pooled records by type:')
    print(df_conv['standard_type'].value_counts().to_string())

    df_conv['standard_value'] = pd.to_numeric(df_conv['standard_value'], errors='coerce')
    df_conv['pchembl_value']  = pd.to_numeric(df_conv['pchembl_value'],  errors='coerce')

    def to_nm(value, unit):
        if pd.isna(value) or value <= 0:
            return np.nan
        return value * {'nM': 1, 'uM': 1e3, 'mM': 1e6, 'M': 1e9}.get(unit, np.nan)

    df_conv['standard_value_nm'] = df_conv.apply(
        lambda r: to_nm(r['standard_value'], r['standard_units']), axis=1
    )

    df_conv['pActivity_calculated'] = -np.log10(df_conv['standard_value_nm'] * 1e-9)
    df_conv['pActivity_row'] = df_conv['pchembl_value'].fillna(df_conv['pActivity_calculated'])

    before = len(df_conv)
    df_conv = df_conv.replace([np.inf, -np.inf], np.nan)
    df_conv = df_conv.dropna(subset=['standard_value_nm', 'pActivity_row'])
    df_conv = df_conv[df_conv['pActivity_row'].between(0, 14)]
    print(f'\nRemoved {before - len(df_conv)} invalid / non-finite rows')
    print(f'Row-level records ready for aggregation: {len(df_conv)}')

    return df_conv

# Keyed by (target, pool) — this is where the 15 combinations diverge
df_step2 = {}
for tk in TARGETS:
    for pool in POOLS:
        print(f'\n>>> {tk.upper()} / {pool} <<<')
        df_step2[(tk, pool)] = convert_and_validate_data(df_step1[tk], allowed_types=POOL_TYPES[pool])


=== STEP 2: Activity Type Selection and pActivity Calculation ===

>>> DRD2 / ki <<<
Activity types available (raw):
standard_type
Ki      13703
EC50     1959
IC50     1575

Pooling ['Ki'] -> 13703 records (79.5% of input)
Breakdown of pooled records by type:
standard_type
Ki    13703

Removed 5 invalid / non-finite rows
Row-level records ready for aggregation: 13698

>>> DRD2 / ki_ic50 <<<
Activity types available (raw):
standard_type
Ki      13703
EC50     1959
IC50     1575

Pooling ['Ki', 'IC50'] -> 15278 records (88.6% of input)
Breakdown of pooled records by type:
standard_type
Ki      13703
IC50     1575

Removed 12 invalid / non-finite rows
Row-level records ready for aggregation: 15266

>>> DRD2 / full <<<
Activity types available (raw):
standard_type
Ki      13703
EC50     1959
IC50     1575

Pooling ['Ki', 'IC50', 'EC50'] -> 17237 records (100.0% of input)
Breakdown of pooled records by type:
standard_type
Ki      13703
EC50     1959
IC50     1575

Removed 21 invalid / non-

In [13]:
# =============================================================================
# STEP 3: Duplicate Aggregation (combined across activity types, per compound)
# =============================================================================

In [14]:
print("\n=== STEP 3: Duplicate Aggregation ===")

def _per_type_stats(df):
    """Vectorized per-activity-type median pActivity/concentration + counts,
    for traceability when Ki/IC50/EC50 get pooled into one endpoint. Median
    commutes with the monotonic -log10 transform, so median(pActivity_row)
    per type == -log10(median(standard_value_nm) per type) — no mismatch."""
    pivot_n = df.pivot_table(index='clean_smiles', columns='standard_type',
                              values='pActivity_row', aggfunc='size', fill_value=0)
    pivot_pact = df.pivot_table(index='clean_smiles', columns='standard_type',
                                 values='pActivity_row', aggfunc='median')
    pivot_conc = df.pivot_table(index='clean_smiles', columns='standard_type',
                                 values='standard_value_nm', aggfunc='median')
    out = pd.DataFrame(index=pivot_n.index)
    for t, suf in [('Ki', 'ki'), ('IC50', 'ic50'), ('EC50', 'ec50')]:
        out[f'n_{suf}']            = pivot_n[t] if t in pivot_n.columns else 0
        out[f'median_p{suf}']      = pivot_pact[t] if t in pivot_pact.columns else np.nan
        out[f'median_{suf}_nm']    = pivot_conc[t] if t in pivot_conc.columns else np.nan
    return out.reset_index()


def handle_duplicates(df, threshold=ACTIVITY_THRESHOLD):
    """
    Aggregate ALL qualifying measurements (per pool) per compound: median
    pActivity across replicate measurements AND across activity types,
    computed after structure standardisation and canonical SMILES-based
    duplicate identification (per project methodology rules).

    The pooled median concentration is named `median_pooled_concentration_nm`
    (not a bare `standard_value_nm`) because for ki_ic50/full pools it may
    combine Ki, IC50, and EC50 measurements — concentrations, but not the same
    pharmacological quantity. Per-type medians/counts are saved alongside so
    pooling effects can be audited later without returning to raw data.
    """
    df_dedup = df.copy()
    df_dedup['clean_smiles'] = df_dedup['clean_smiles'].astype(str)

    n_compounds_before = df_dedup['clean_smiles'].nunique()
    print(f'Row-level records            : {len(df_dedup)}')
    print(f'Unique compounds             : {n_compounds_before}')

    df_agg = df_dedup.groupby('clean_smiles').agg(
        molecule_chembl_id              = ('molecule_chembl_id', 'first'),
        pActivity                       = ('pActivity_row', 'median'),
        median_pooled_concentration_nm  = ('standard_value_nm', 'median'),
        n_measurements                  = ('pActivity_row', 'size'),
        n_assays                        = ('assay_chembl_id', lambda x: len(set(x))),
        n_activity_types                = ('standard_type', lambda x: len(set(x))),
        activity_types_used             = ('standard_type', lambda x: '; '.join(sorted(set(x)))),
    ).reset_index()

    df_agg = df_agg.merge(_per_type_stats(df_dedup), on='clean_smiles', how='left')

    df_agg['activity'] = (df_agg['pActivity'] >= threshold).astype(np.int8)

    print(f'After aggregation             : {len(df_agg)} compounds')
    print(f'\nMeasurements-per-compound     : median={df_agg["n_measurements"].median():.0f}, '
          f'max={df_agg["n_measurements"].max():.0f}')
    single_assay_pct = (df_agg['n_assays'] == 1).mean() * 100
    print(f'Single-assay compounds        : {single_assay_pct:.1f}%')
    print(f'\nThreshold: pActivity >= {threshold}')
    print('Activity distribution:')
    print(df_agg['activity'].value_counts())

    return df_agg

df_step3 = {}
for tk in TARGETS:
    for pool in POOLS:
        print(f'\n>>> {tk.upper()} / {pool} <<<')
        df_step3[(tk, pool)] = handle_duplicates(df_step2[(tk, pool)])



=== STEP 3: Duplicate Aggregation ===

>>> DRD2 / ki <<<
Row-level records            : 13698
Unique compounds             : 9787
After aggregation             : 9787 compounds

Measurements-per-compound     : median=1, max=105
Single-assay compounds        : 78.0%

Threshold: pActivity >= 6.0
Activity distribution:
activity
1    7022
0    2765
Name: count, dtype: int64

>>> DRD2 / ki_ic50 <<<
Row-level records            : 15266
Unique compounds             : 10685
After aggregation             : 10685 compounds

Measurements-per-compound     : median=1, max=115
Single-assay compounds        : 77.1%

Threshold: pActivity >= 6.0
Activity distribution:
activity
1    7467
0    3218
Name: count, dtype: int64

>>> DRD2 / full <<<
Row-level records            : 17216
Unique compounds             : 10934
After aggregation             : 10934 compounds

Measurements-per-compound     : median=1, max=115
Single-assay compounds        : 74.1%

Threshold: pActivity >= 6.0
Activity distribution:


In [15]:
# =============================================================================
# STEP 4: Missing Data Treatment
# =============================================================================

In [16]:
print("\n=== STEP 4: Missing Data Treatment ===")

def handle_missing_data(df):
    """Drop rows missing essential columns."""
    df_missing = df.copy()

    print('Missing data summary:')
    missing_summary = df_missing.isnull().sum()
    print(missing_summary[missing_summary > 0])

    essential_cols = ['clean_smiles', 'median_pooled_concentration_nm', 'pActivity']
    df_missing = df_missing.dropna(subset=essential_cols)
    print(f'After removing rows with missing essential data: {len(df_missing)} records')

    return df_missing

df_step4 = {}
for tk in TARGETS:
    for pool in POOLS:
        print(f'\n>>> {tk.upper()} / {pool} <<<')
        df_step4[(tk, pool)] = handle_missing_data(df_step3[(tk, pool)])



=== STEP 4: Missing Data Treatment ===

>>> DRD2 / ki <<<
Missing data summary:
median_pic50      9787
median_ic50_nm    9787
median_pec50      9787
median_ec50_nm    9787
dtype: int64
After removing rows with missing essential data: 9787 records

>>> DRD2 / ki_ic50 <<<
Missing data summary:
median_pki          898
median_ki_nm        898
median_pic50       9434
median_ic50_nm     9434
median_pec50      10685
median_ec50_nm    10685
dtype: int64
After removing rows with missing essential data: 10685 records

>>> DRD2 / full <<<
Missing data summary:
median_pki         1147
median_ki_nm       1147
median_pic50       9683
median_ic50_nm     9683
median_pec50      10062
median_ec50_nm    10062
dtype: int64
After removing rows with missing essential data: 10934 records

>>> CB2 / ki <<<
Missing data summary:
median_pic50      5099
median_ic50_nm    5099
median_pec50      5099
median_ec50_nm    5099
dtype: int64
After removing rows with missing essential data: 5099 records

>>> CB2 / ki_ic

In [17]:
# =============================================================================
# STEP 5: Data Quality Filtering
# =============================================================================

In [18]:
print("\n=== STEP 5: Data Quality Filtering ===")

def quality_filtering(df):
    """Filter records to biologically plausible activity ranges using FIXED,
    pre-defined bounds only (0.1 nM - 1 mM concentration, pActivity 3.0-12.0)
    — identical across all targets/pools, per the project's pipeline-
    consistency requirement.

    IQR-based outlier removal is intentionally NOT applied as a filter: it
    would compute a different accepted pActivity range per target/pool (each
    has its own quartiles), silently varying the pipeline per target, and it
    would also remove genuinely high-affinity ligands or valid inactives for
    being statistically extreme rather than biologically implausible. Instead
    it's computed and kept as a non-destructive flag (`pActivity_iqr_outlier`)
    so it can be reported/inspected without altering the modelling dataset.
    """
    df_quality = df.copy()

    # Typical range for small-molecule bioactivity data: 0.1 nM - 1 mM
    before_range = len(df_quality)
    df_quality = df_quality[
        (df_quality['median_pooled_concentration_nm'] >= 0.1) &
        (df_quality['median_pooled_concentration_nm'] <= 1_000_000)
    ]
    print(f'Removed {before_range - len(df_quality)} records outside activity range (0.1 nM - 1 mM)')

    before_pact_range = len(df_quality)
    df_quality = df_quality[
        (df_quality['pActivity'] >= 3.0) &
        (df_quality['pActivity'] <= 12.0)
    ]
    print(f'Removed {before_pact_range - len(df_quality)} records outside pActivity range (3.0 - 12.0)')

    Q1 = df_quality['pActivity'].quantile(0.25)
    Q3 = df_quality['pActivity'].quantile(0.75)
    IQR = Q3 - Q1
    df_quality['pActivity_iqr_outlier'] = ~df_quality['pActivity'].between(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    n_flagged = int(df_quality['pActivity_iqr_outlier'].sum())
    print(f'Flagged (NOT removed) {n_flagged} statistical IQR outliers '
          f'({n_flagged / len(df_quality) * 100:.1f}%) — see pActivity_iqr_outlier column')

    return df_quality

df_step5 = {}
for tk in TARGETS:
    for pool in POOLS:
        print(f'\n>>> {tk.upper()} / {pool} <<<')
        df_step5[(tk, pool)] = quality_filtering(df_step4[(tk, pool)])



=== STEP 5: Data Quality Filtering ===

>>> DRD2 / ki <<<
Removed 12 records outside activity range (0.1 nM - 1 mM)
Removed 0 records outside pActivity range (3.0 - 12.0)
Flagged (NOT removed) 113 statistical IQR outliers (1.2%) — see pActivity_iqr_outlier column

>>> DRD2 / ki_ic50 <<<
Removed 13 records outside activity range (0.1 nM - 1 mM)
Removed 0 records outside pActivity range (3.0 - 12.0)
Flagged (NOT removed) 89 statistical IQR outliers (0.8%) — see pActivity_iqr_outlier column

>>> DRD2 / full <<<
Removed 12 records outside activity range (0.1 nM - 1 mM)
Removed 0 records outside pActivity range (3.0 - 12.0)
Flagged (NOT removed) 73 statistical IQR outliers (0.7%) — see pActivity_iqr_outlier column

>>> CB2 / ki <<<
Removed 21 records outside activity range (0.1 nM - 1 mM)
Removed 0 records outside pActivity range (3.0 - 12.0)
Flagged (NOT removed) 0 statistical IQR outliers (0.0%) — see pActivity_iqr_outlier column

>>> CB2 / ki_ic50 <<<
Removed 21 records outside activity

In [19]:
# =============================================================================
# STEP 6: PAINS and Structural Liability Annotation
# =============================================================================

In [20]:
print("\n=== STEP 6: PAINS and Structural Liability (Brenk) Annotation ===")

def annotate_pains(df, smiles_col='clean_smiles'):
    """Annotate PAINS and Brenk alerts SEPARATELY (non-destructive — compounds
    kept). Previously these were merged into one FilterCatalog, so `is_pains`
    actually meant "PAINS or Brenk" — a real labeling bug (reported
    PAINS-free% was wrong). Two independent catalogs/flags now."""
    df_ann = df.copy()
    block = BlockLogs()

    pains_params = FilterCatalog.FilterCatalogParams()
    pains_params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    pains_catalog = FilterCatalog.FilterCatalog(pains_params)

    brenk_params = FilterCatalog.FilterCatalogParams()
    brenk_params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.BRENK)
    brenk_catalog = FilterCatalog.FilterCatalog(brenk_params)

    pains_flags, pains_reasons, n_pains_list = [], [], []
    brenk_flags, brenk_reasons, n_brenk_list = [], [], []
    for smi in tqdm(df_ann[smiles_col], desc='Structural alert annotation'):
        mol = Chem.MolFromSmiles(smi)
        p_reasons = [m.GetDescription() for m in pains_catalog.GetMatches(mol)] if mol else []
        b_reasons = [m.GetDescription() for m in brenk_catalog.GetMatches(mol)] if mol else []
        pains_flags.append(bool(p_reasons)); pains_reasons.append('; '.join(p_reasons)); n_pains_list.append(len(p_reasons))
        brenk_flags.append(bool(b_reasons)); brenk_reasons.append('; '.join(b_reasons)); n_brenk_list.append(len(b_reasons))
    del block

    df_ann['is_pains']       = pains_flags
    df_ann['pains_reason']   = pains_reasons
    df_ann['n_pains_alerts'] = n_pains_list
    df_ann['is_brenk']       = brenk_flags
    df_ann['brenk_reason']   = brenk_reasons
    df_ann['n_brenk_alerts'] = n_brenk_list
    df_ann['has_any_structural_alert'] = df_ann['is_pains'] | df_ann['is_brenk']

    n = len(df_ann)
    print(f'PAINS flagged : {df_ann["is_pains"].sum()} / {n} ({df_ann["is_pains"].sum()/n*100:.1f}%)')
    print(f'Brenk flagged : {df_ann["is_brenk"].sum()} / {n} ({df_ann["is_brenk"].sum()/n*100:.1f}%)')
    print(f'Any alert     : {df_ann["has_any_structural_alert"].sum()} / {n} ({df_ann["has_any_structural_alert"].sum()/n*100:.1f}%)')
    print('Note: flagged compounds retained; use is_pains/is_brenk columns to filter downstream.')

    return df_ann

df_step6 = {}
for tk in TARGETS:
    for pool in POOLS:
        print(f'\n>>> {tk.upper()} / {pool} <<<')
        df_step6[(tk, pool)] = annotate_pains(df_step5[(tk, pool)])



=== STEP 6: PAINS and Structural Liability (Brenk) Annotation ===

>>> DRD2 / ki <<<


Structural alert annotation: 100%|██████████| 9775/9775 [00:36<00:00, 267.03it/s]


PAINS flagged : 326 / 9775 (3.3%)
Brenk flagged : 4250 / 9775 (43.5%)
Any alert     : 4408 / 9775 (45.1%)
Note: flagged compounds retained; use is_pains/is_brenk columns to filter downstream.

>>> DRD2 / ki_ic50 <<<


Structural alert annotation: 100%|██████████| 10672/10672 [00:41<00:00, 257.29it/s]


PAINS flagged : 338 / 10672 (3.2%)
Brenk flagged : 4619 / 10672 (43.3%)
Any alert     : 4785 / 10672 (44.8%)
Note: flagged compounds retained; use is_pains/is_brenk columns to filter downstream.

>>> DRD2 / full <<<


...

PAINS flagged : 0 / 158 (0.0%)
Brenk flagged : 98 / 158 (62.0%)
Any alert     : 98 / 158 (62.0%)
Note: flagged compounds retained; use is_pains/is_brenk columns to filter downstream.

>>> CCR5 / ki_ic50 <<<


Structural alert annotation: 100%|██████████| 2396/2396 [00:10<00:00, 238.58it/s]


PAINS flagged : 67 / 2396 (2.8%)
Brenk flagged : 651 / 2396 (27.2%)
Any alert     : 685 / 2396 (28.6%)
Note: flagged compounds retained; use is_pains/is_brenk columns to filter downstream.

>>> CCR5 / full <<<


Structural alert annotation: 100%|██████████| 2435/2435 [00:10<00:00, 222.78it/s]

PAINS flagged : 79 / 2435 (3.2%)
Brenk flagged : 683 / 2435 (28.0%)
Any alert     : 717 / 2435 (29.4%)
Note: flagged compounds retained; use is_pains/is_brenk columns to filter downstream.


In [21]:
# =============================================================================
# STEP 7: Molecular Descriptor Annotation
# =============================================================================

In [22]:
print("\n=== STEP 7: Molecular Descriptor Annotation ===")

def annotate_descriptors(df, mol_col='clean_smiles'):
    """Compute RDKit physicochemical descriptors and drug-likeness flags."""
    df_desc = df.copy()
    block = BlockLogs()

    mols = [
        Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
        for smi in df_desc[mol_col]
    ]

    df_desc['MW']       = [Descriptors.MolWt(m)             if m else np.nan for m in mols]
    df_desc['LogP']     = [Crippen.MolLogP(m)               if m else np.nan for m in mols]
    df_desc['TPSA']     = [Descriptors.TPSA(m)              if m else np.nan for m in mols]
    df_desc['HBD']      = [Lipinski.NumHDonors(m)           if m else np.nan for m in mols]
    df_desc['HBA']      = [Lipinski.NumHAcceptors(m)        if m else np.nan for m in mols]
    df_desc['RotBonds'] = [Descriptors.NumRotatableBonds(m) if m else np.nan for m in mols]

    # Non-redundant additions for chemical-space characterisation (Q2). Skipped
    # NHOHCount/NOCount (near-duplicates of HBD/HBA) and FormalCharge (near-
    # uniformly 0 post Step-1 uncharging, not informative here).
    df_desc['HeavyAtomCount']    = [Descriptors.HeavyAtomCount(m)  if m else np.nan for m in mols]
    df_desc['RingCount']         = [Descriptors.RingCount(m)       if m else np.nan for m in mols]
    df_desc['AromaticRingCount'] = [Descriptors.NumAromaticRings(m) if m else np.nan for m in mols]
    df_desc['FractionCSP3']      = [Descriptors.FractionCSP3(m)    if m else np.nan for m in mols]
    del block

    df_desc['Lipinski_compliant'] = (
        (df_desc['MW']  <= 500) &
        (df_desc['LogP'] <= 5)  &
        (df_desc['HBD'] <= 5)   &
        (df_desc['HBA'] <= 10)
    )
    df_desc['DrugLike_soft'] = (
        df_desc['MW'].between(150, 800) &
        df_desc['LogP'].between(-3, 6)  &
        (df_desc['TPSA'] <= 200)        &
        (df_desc['RotBonds'] <= 15)
    )

    print(f'Lipinski compliant : {df_desc["Lipinski_compliant"].sum()} / {len(df_desc)}')
    print(f'Soft drug-like     : {df_desc["DrugLike_soft"].sum()} / {len(df_desc)}')
    return df_desc

df_step7 = {}
for tk in TARGETS:
    for pool in POOLS:
        print(f'\n>>> {tk.upper()} / {pool} <<<')
        df_step7[(tk, pool)] = annotate_descriptors(df_step6[(tk, pool)])



=== STEP 7: Molecular Descriptor Annotation ===

>>> DRD2 / ki <<<
Lipinski compliant : 7003 / 9775
Soft drug-like     : 9161 / 9775

>>> DRD2 / ki_ic50 <<<
Lipinski compliant : 7753 / 10672
Soft drug-like     : 10007 / 10672

>>> DRD2 / full <<<
Lipinski compliant : 7952 / 10922
Soft drug-like     : 10233 / 10922

>>> CB2 / ki <<<
Lipinski compliant : 2175 / 5078
Soft drug-like     : 3634 / 5078

>>> CB2 / ki_ic50 <<<
Lipinski compliant : 2772 / 6402
Soft drug-like     : 4554 / 6402

>>> CB2 / full <<<
Lipinski compliant : 5297 / 9910
Soft drug-like     : 7417 / 9910

>>> ADORA2A / ki <<<
Lipinski compliant : 5334 / 6563
Soft drug-like     : 6398 / 6563

>>> ADORA2A / ki_ic50 <<<
Lipinski compliant : 6749 / 8171
Soft drug-like     : 7970 / 8171

>>> ADORA2A / full <<<
Lipinski compliant : 6848 / 8317
Soft drug-like     : 8096 / 8317

>>> OPRM1 / ki <<<
Lipinski compliant : 3687 / 6289
Soft drug-like     : 5406 / 6289

>>> OPRM1 / ki_ic50 <<<
Lipinski compliant : 4763 / 7834
Soft drug

In [23]:
# =============================================================================
# STEP 8: Assay Support Annotation
# =============================================================================

In [24]:
print("\n=== STEP 8: Assay Support Annotation ===")

def annotate_assay_support(df, min_assays=1):
    """Annotate compounds by number of supporting assays."""
    df_out = df.copy()
    print(f'Total records: {len(df_out)}')

    if 'n_assays' not in df_out.columns:
        raise ValueError("Column 'n_assays' missing — run handle_duplicates first")

    print('\nAssay count distribution:')
    print(df_out['n_assays'].value_counts().sort_index())
    print(f'Median assays per compound: {df_out["n_assays"].median():.0f}')

    df_out['assay_support'] = pd.cut(
        df_out['n_assays'],
        bins=[0, 1, 3, 10, np.inf],
        labels=['single', 'limited', 'moderate', 'strong'],
        include_lowest=True,
    )

    before = len(df_out)
    if min_assays > 1:
        df_out = df_out[df_out['n_assays'] >= min_assays]
        print(f'\nAfter filtering (n_assays >= {min_assays}): {len(df_out)} records ({before - len(df_out)} removed)')

    print('\nAssay support summary:')
    print(df_out['assay_support'].value_counts().sort_index())
    return df_out

df_step8 = {}
for tk in TARGETS:
    for pool in POOLS:
        print(f'\n>>> {tk.upper()} / {pool} <<<')
        df_step8[(tk, pool)] = annotate_assay_support(df_step7[(tk, pool)], min_assays=1)


=== STEP 8: Assay Support Annotation ===

>>> DRD2 / ki <<<
Total records: 9775

Assay count distribution:
n_assays
1      7625
2      1825
3       194
4        51
5        24
6        14
7        13
8         4
9         4
10        3
12        1
13        1
14        2
15        1
16        3
18        1
20        1
21        1
22        1
29        2
36        1
41        1
78        1
102       1
Name: count, dtype: int64
Median assays per compound: 1

Assay support summary:
assay_support
single      7625
limited     2019
moderate     113
strong        18
Name: count, dtype: int64

>>> DRD2 / ki_ic50 <<<
Total records: 10672

Assay count distribution:
n_assays
1      8226
2      2049
3       219
4        76
5        36
6        17
7        14
8         8
9         3
10        5
14        3
15        3
16        2
17        1
19        1
23        2
27        1
29        1
31        1
36        1
48        1
88        1
110       1
Name: count, dtype: int64
Median assays per compou

In [25]:
# =============================================================================
# STEP 9: Final Data Validation
# =============================================================================

In [26]:
print("\n=== STEP 9: Final Data Validation ===")

def final_validation(df):
    """Sanity-check the cleaned dataset and print a quality report."""
    df_final = df.copy()

    print(f'Total records         : {len(df_final)}')
    print(f'Unique compounds      : {df_final["clean_smiles"].nunique()}')

    if 'median_pooled_concentration_nm' in df_final.columns:
        print(f'Potency range (nM)    : {df_final["median_pooled_concentration_nm"].min():.2f} - {df_final["median_pooled_concentration_nm"].max():.2f}')
    if 'pActivity' in df_final.columns:
        print(f'pActivity range       : {df_final["pActivity"].min():.2f} - {df_final["pActivity"].max():.2f}')

    print('\nMolecular properties summary:')
    prop_cols = ['MW', 'LogP', 'TPSA', 'HBD', 'HBA', 'RotBonds',
                 'HeavyAtomCount', 'RingCount', 'AromaticRingCount', 'FractionCSP3']
    print(df_final[prop_cols].describe())

    print('\nMissing values in final dataset:')
    missing_final = df_final.isnull().sum()
    print(missing_final[missing_final > 0] if missing_final.any() else 'None')

    if 'Lipinski_compliant' in df_final.columns:
        n = df_final['Lipinski_compliant'].sum()
        print(f'\nLipinski compliant : {n} / {len(df_final)} ({n/len(df_final)*100:.1f}%)')

    if 'is_pains' in df_final.columns:
        pains_free = (~df_final['is_pains']).sum()
        print(f'PAINS-free         : {pains_free} / {len(df_final)} ({pains_free/len(df_final)*100:.1f}%)')
    if 'is_brenk' in df_final.columns:
        brenk_free = (~df_final['is_brenk']).sum()
        print(f'Brenk-free         : {brenk_free} / {len(df_final)} ({brenk_free/len(df_final)*100:.1f}%)')
    if 'has_any_structural_alert' in df_final.columns:
        alert_free = (~df_final['has_any_structural_alert']).sum()
        print(f'Any-alert-free     : {alert_free} / {len(df_final)} ({alert_free/len(df_final)*100:.1f}%)')

    completeness = ((1 - df_final.isnull().sum() / len(df_final)) * 100).mean()
    print(f'\nAvg column completeness: {completeness:.1f}%')

    return df_final

df_cleaned = {}
for tk in TARGETS:
    for pool in POOLS:
        print(f'\n>>> {tk.upper()} / {pool} <<<')
        df_cleaned[(tk, pool)] = final_validation(df_step8[(tk, pool)])



=== STEP 9: Final Data Validation ===

>>> DRD2 / ki <<<
Total records         : 9775
Unique compounds      : 9775
Potency range (nM)    : 0.10 - 1000000.00
pActivity range       : 3.00 - 10.00

Molecular properties summary:
                MW         LogP         TPSA          HBD          HBA  \
count  9775.000000  9775.000000  9775.000000  9775.000000  9775.000000   
mean    431.259085     4.059353    57.939150     1.040409     4.599693   
std     165.368953     1.345920    55.443546     1.921176     2.482404   
min     135.166000   -10.732830     3.240000     0.000000     0.000000   
25%     362.864500     3.268650    36.610000     0.000000     3.000000   
50%     422.481000     4.037500    54.040000     1.000000     4.000000   
75%     477.611000     4.797050    69.310000     1.000000     6.000000   
max    3771.262000    16.768600  1367.990000    41.000000    58.000000   

          RotBonds  HeavyAtomCount    RingCount  AromaticRingCount  \
count  9775.000000     9775.000000  9

In [27]:
# =============================================================================
# STEP 10: Summary
# =============================================================================

In [28]:
print("\n=== FINAL CLEANED DATASET SUMMARY ===")

def final_summary_table(df, n_raw, activity_pool, allowed_types):
    """Print a concise summary of the cleaned dataset. activity_pool/allowed_types
    passed explicitly (not read from a module-level global) since this now runs
    once per (target, pool) combination, not once per notebook run."""
    total = len(df)
    unique = df['clean_smiles'].nunique()
    retention = total / n_raw * 100

    active = (df['activity'] == 1).sum() if 'activity' in df.columns else None
    inactive = total - active if active is not None else None

    pa = df['pActivity'] if 'pActivity' in df.columns else None

    lip = df['Lipinski_compliant'].sum() if 'Lipinski_compliant' in df.columns else None
    dl  = df['DrugLike_soft'].sum()      if 'DrugLike_soft'      in df.columns else None
    pf  = (~df['is_pains']).sum()        if 'is_pains'          in df.columns else None

    single_assay_pct = (df['n_assays'] == 1).mean() * 100 if 'n_assays' in df.columns else None
    completeness = ((1 - df.isnull().sum() / total) * 100).mean()

    summary = {
        'Original records'                 : n_raw,
        'Final records (unique compounds)' : total,
        'Data retention (%)'               : f'{retention:.1f}',
        'Unique compounds'                 : unique,
        '── Activity ──'                  : '─────────',
        'Activity pool'                    : activity_pool,
        'Types pooled'                     : ', '.join(allowed_types),
        'Active threshold'                 : f'pActivity >= {ACTIVITY_THRESHOLD}',
        'Active compounds'                 : active,
        'Inactive compounds'               : inactive,
        'pActivity range'                  : f'{pa.min():.2f} - {pa.max():.2f}' if pa is not None else 'N/A',
        'pActivity mean ± std'             : f'{pa.mean():.2f} ± {pa.std():.2f}' if pa is not None else 'N/A',
        'Measurements/compound (median, max)' : f'{df["n_measurements"].median():.0f}, {df["n_measurements"].max():.0f}' if 'n_measurements' in df.columns else 'N/A',
        'Single-assay compounds (%)'        : f'{single_assay_pct:.1f}' if single_assay_pct is not None else 'N/A',
        '── Quality ──'                   : '─────────',
        'Lipinski compliant (%)'           : f'{lip/total*100:.1f}' if lip is not None else 'N/A',
        'Soft drug-like (%)'               : f'{dl/total*100:.1f}'  if dl  is not None else 'N/A',
        'PAINS-free (%)'                    : f'{pf/total*100:.1f}'  if pf  is not None else 'N/A',
        'Avg completeness (%)'              : f'{completeness:.1f}',
    }

    summary_df = pd.DataFrame(summary, index=[0]).T.rename(columns={0: 'Value'})
    print(summary_df)
    return summary_df

summary_df = {}
for tk in TARGETS:
    for pool in POOLS:
        print(f'\n>>> {tk.upper()} / {pool} <<<')
        summary_df[(tk, pool)] = final_summary_table(
            df_cleaned[(tk, pool)], n_raw=len(df_raw[tk]),
            activity_pool=pool, allowed_types=POOL_TYPES[pool],
        )


=== FINAL CLEANED DATASET SUMMARY ===

>>> DRD2 / ki <<<
                                                Value
Original records                                17263
Final records (unique compounds)                 9775
Data retention (%)                               56.6
Unique compounds                                 9775
── Activity ──                              ─────────
Activity pool                                      ki
Types pooled                                       Ki
Active threshold                     pActivity >= 6.0
Active compounds                                 7015
Inactive compounds                               2760
pActivity range                          3.00 - 10.00
pActivity mean ± std                      6.56 ± 1.07
Measurements/compound (median, max)            1, 105
Single-assay compounds (%)                       78.0
── Quality ──                               ─────────
Lipinski compliant (%)                           71.6
Soft drug-like (%)      

## Assertions: fail fast if pipeline assumptions are violated

Checked per (target, pool): uniqueness/nullness of key columns, activity
label consistency with the threshold, measurement/assay/type-count sanity,
pActivity bounds, and activity-type membership. Pool-nesting is checked (and
warned, not hard-failed — see cross-pool overlap cell above) since minor
edge cases there are diagnostic, not necessarily fatal.

In [29]:
print("=== ASSERTIONS: fail fast if pipeline assumptions are violated ===")

assertion_failures = []
for tk in TARGETS:
    for pool in POOLS:
        df = df_cleaned[(tk, pool)]
        label = f'{tk}/{pool}'
        checks = [
            ("clean_smiles unique", df['clean_smiles'].is_unique),
            ("clean_smiles not null", df['clean_smiles'].notna().all()),
            ("pActivity not null", df['pActivity'].notna().all()),
            ("activity in {0,1}", df['activity'].isin([0, 1]).all()),
            ("activity matches threshold", np.array_equal(
                df['activity'].to_numpy(),
                (df['pActivity'] >= ACTIVITY_THRESHOLD).astype(np.int8).to_numpy())),
            ("n_measurements >= 1", df['n_measurements'].ge(1).all()),
            ("n_assays >= 1", df['n_assays'].ge(1).all()),
            ("n_activity_types >= 1", df['n_activity_types'].ge(1).all()),
            ("pActivity within [3, 12]", df['pActivity'].between(3, 12).all()),
        ]
        for name, ok in checks:
            if not ok:
                assertion_failures.append(f'{label}: FAILED "{name}"')

        used_types = set(
            t for s in df['activity_types_used'].dropna() for t in s.split('; ') if t
        )
        allowed = set(POOL_TYPES[pool])
        if not used_types <= allowed:
            assertion_failures.append(f'{label}: activity_types_used has types outside {allowed}: {used_types - allowed}')

if assertion_failures:
    print(f'{len(assertion_failures)} ASSERTION FAILURE(S):')
    for f in assertion_failures:
        print(f'  {f}')
    raise AssertionError(f'{len(assertion_failures)} pipeline assumption(s) violated — see printed list above')
else:
    print(f'All assertions passed for all {len(TARGETS) * len(POOLS)} (target, pool) combinations.')

# CCR5-ki explicit low-N warning — do not silently send to modelling
_ccr5_ki = df_cleaned.get(('ccr5', 'ki'))
if _ccr5_ki is not None:
    _n = len(_ccr5_ki)
    _n_inactive = int((_ccr5_ki['activity'] == 0).sum())
    if _n < 500 or _n_inactive < 20:
        print(f"\nWARNING: CCR5/ki has {_n} compounds ({_n_inactive} inactive) — "
              f"insufficient for reliable binary classification / scaffold-split validation. "
              f"Exclude from notebook 03's per-pool sensitivity comparison explicitly — "
              f"do not silently include it.")


=== ASSERTIONS: fail fast if pipeline assumptions are violated ===
All assertions passed for all 15 (target, pool) combinations.



In [30]:
# Drop intermediate calculated column before saving
df_save = {}
output_filename = {}
for tk in TARGETS:
    for pool in POOLS:
        cols_to_drop = [c for c in ['pActivity_calculated', 'pActivity_row'] if c in df_cleaned[(tk, pool)].columns]
        df_save[(tk, pool)] = df_cleaned[(tk, pool)].drop(columns=cols_to_drop)

        output_filename[(tk, pool)] = f'cleaned_data_{tk}_{pool}.csv'
        df_save[(tk, pool)].to_csv(processed_path / output_filename[(tk, pool)], index=False)
        print(f'{tk.upper()}/{pool} cleaned dataset saved: {processed_path / output_filename[(tk, pool)]}')


def _capture_package_versions():
    """Best-effort __version__ capture for the packages THIS notebook
    actually imports — targeted, unlike the noisy full `pip freeze`."""
    packages = ['rdkit', 'numpy', 'pandas', 'sklearn']
    versions = {}
    for pkg in packages:
        try:
            mod = __import__(pkg)
            versions[pkg] = getattr(mod, '__version__', 'unknown')
        except ImportError:
            pass
    return versions


def _hash_file(path, chunk_size=1 << 20):
    """SHA-256 of a file, chunked so large CSVs don't blow up memory."""
    import hashlib
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()


DRD2/ki cleaned dataset saved: /content/drive/My Drive/gpcr_benchmark/data/processed/cleaned_data_drd2_ki.csv
DRD2/ki_ic50 cleaned dataset saved: /content/drive/My Drive/gpcr_benchmark/data/processed/cleaned_data_drd2_ki_ic50.csv
DRD2/full cleaned dataset saved: /content/drive/My Drive/gpcr_benchmark/data/processed/cleaned_data_drd2_full.csv
CB2/ki cleaned dataset saved: /content/drive/My Drive/gpcr_benchmark/data/processed/cleaned_data_cb2_ki.csv
CB2/ki_ic50 cleaned dataset saved: /content/drive/My Drive/gpcr_benchmark/data/processed/cleaned_data_cb2_ki_ic50.csv
CB2/full cleaned dataset saved: /content/drive/My Drive/gpcr_benchmark/data/processed/cleaned_data_cb2_full.csv
ADORA2A/ki cleaned dataset saved: /content/drive/My Drive/gpcr_benchmark/data/processed/cleaned_data_adora2a_ki.csv
ADORA2A/ki_ic50 cleaned dataset saved: /content/drive/My Drive/gpcr_benchmark/data/processed/cleaned_data_adora2a_ki_ic50.csv
ADORA2A/full cleaned dataset saved: /content/drive/My Drive/gpcr_benchmark/d

In [31]:
def write_manifest(manifest_path, config_summary, outputs, inputs=None):
    """Single source-of-truth record of config + inputs + outputs + timestamp
    (+ best-effort git commit, Python version, SHA-256 of every input AND
    output file) for this run — see notebook 01 for rationale. Hashing lets a
    later notebook (or a reviewer) verify it's reading the exact files this
    run produced/consumed, not stale/hand-edited copies."""
    import datetime, subprocess, platform, json as _json
    git_hash = None
    try:
        git_hash = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], stderr=subprocess.DEVNULL, cwd=str(PROJECT_DIR)
        ).decode().strip()
    except Exception:
        pass

    def _hash_registry(registry):
        hashed = {}
        for name, meta in registry.items():
            meta = dict(meta)
            p = Path(meta.get('path', ''))
            if p.exists() and p.is_file():
                try:
                    meta['sha256'] = _hash_file(p)
                except Exception:
                    pass
            hashed[name] = meta
        return hashed

    manifest = {
        'timestamp': datetime.datetime.now().isoformat(),
        'python_version': platform.python_version(),
        'git_commit': git_hash,
        'config': config_summary,
        'inputs': _hash_registry(inputs) if inputs else {},
        'outputs': _hash_registry(outputs),
        'package_versions': _capture_package_versions(),
    }
    with open(manifest_path, 'w') as f:
        _json.dump(manifest, f, indent=2, default=str)
    n_in = len(manifest['inputs'])
    n_out = len(manifest['outputs'])
    print(f'Manifest saved: {manifest_path} ({n_in} inputs, {n_out} outputs hashed)')
    return manifest


## Dataset Characterisation: Tables (all 5 targets x 3 pools)

Everything below persists tables in final, machine-readable form — not just
printed — following the project's over-generate-now / triage-at-write-up-time
convention: main-text vs. supplementary placement is a later decision, not a
reason to skip computing something now.

`all_outputs` accumulates every file this section saves, so the manifest at
the end doesn't need a hand-maintained list.

In [32]:
# =============================================================================
# PUBLICATION FIGURE STYLE — no on-figure titles (the manuscript caption
# carries that), no code identifiers in any label/legend, American
# spelling, 300dpi PNG + vector PDF.
# =============================================================================
import matplotlib as mpl

mpl.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
BLUE, GREY, RED, ORANGE = "#1f77b4", "#c9c9c9", "#d62728", "#ff7f0e"
# Okabe-Ito colorblind-safe palette, one color per target
TARGET_COLORS = dict(zip(TARGETS, ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7"]))

# Human-readable labels — every pool/descriptor key that ends up as figure
# text (axis tick, legend entry, panel label) goes through one of these, not
# the raw code identifier.
POOL_LABELS = {'ki': 'Ki', 'ki_ic50': 'Ki + IC50', 'full': 'Ki + IC50 + EC50'}
DESCRIPTOR_LABELS = {
    'MW': 'Molecular weight (Da)', 'LogP': 'LogP', 'TPSA': 'TPSA (Å²)',
    'HBD': 'H-bond donors', 'HBA': 'H-bond acceptors', 'RotBonds': 'Rotatable bonds',
    'HeavyAtomCount': 'Heavy atom count', 'RingCount': 'Ring count',
    'AromaticRingCount': 'Aromatic ring count', 'FractionCSP3': 'Fsp³',
}

figures_path = processed_path / "figures"
figures_path.mkdir(parents=True, exist_ok=True)

def save_fig(fig, stem):
    for ext in ("png", "pdf"):
        fig.savefig(figures_path / f"{stem}.{ext}")
    plt.close(fig)
    print(f"Figure saved: {figures_path / stem}.png / .pdf")

def panel_label(ax, letter):
    """Bold panel letter (A/B/C...), top-left — NOT a descriptive title.
    The manuscript caption carries the description; the letter alone is
    what lets the caption cross-reference each panel."""
    ax.set_title(letter, loc='left', fontweight='bold', fontsize=12)

all_outputs = {}

def _register(name, path_obj, n_rows=None):
    entry = {'path': str(path_obj)}
    if n_rows is not None:
        entry['n_rows'] = n_rows
    all_outputs[name] = entry

def register_figure(stem):
    """Register BOTH the PNG and PDF save_fig() writes for a figure —
    a figure-cell that only registered the .png left the .pdf unhashed
    in the manifest even though save_fig() wrote it."""
    for ext in ('png', 'pdf'):
        filename = f'{stem}.{ext}'
        _register(filename, figures_path / filename)


In [33]:
print("=== STRUCTURE-CLEANING EXCLUSION SUMMARY (Step 1, per target) ===")
print("(coarse 2-category split: missing SMILES vs failed standardisation —")
print(" see clean_structures() docstring for why a deeper reason taxonomy isn't captured)")

if structure_exclusion_records:
    structure_exclusion_df = pd.DataFrame(structure_exclusion_records)
else:
    # USE_CACHED_STEP1 was True — no fresh exclusion stats this run
    structure_exclusion_df = pd.DataFrame(columns=[
        'target', 'initial_records', 'missing_smiles_removed',
        'failed_standardisation_removed', 'final_valid_structures'])
    print('(USE_CACHED_STEP1=True this run — no fresh exclusion stats to report)')

print(structure_exclusion_df.to_string(index=False))

structure_exclusion_path = processed_path / 'structure_cleaning_exclusion_summary.csv'
structure_exclusion_df.to_csv(structure_exclusion_path, index=False)
_register('structure_cleaning_exclusion_summary.csv', structure_exclusion_path, len(structure_exclusion_df))
print(f'\nSaved: {structure_exclusion_path}')


=== STRUCTURE-CLEANING EXCLUSION SUMMARY (Step 1, per target) ===
(coarse 2-category split: missing SMILES vs failed standardisation —
 see clean_structures() docstring for why a deeper reason taxonomy isn't captured)
 initial_records  missing_smiles_removed  failed_standardisation_removed  final_valid_structures  target
           17263                      26                               0                   17237    drd2
           13662                       1                               0                   13661     cb2
           10554                       3                               0                   10551 adora2a
           14009                      37                               0                   13972   oprm1
            3449                       0                               0                    3449    ccr5

Saved: /content/drive/My Drive/gpcr_benchmark/data/processed/structure_cleaning_exclusion_summary.csv


In [34]:
print("=== CLEANING ATTRITION LOG (long format, all targets x pools) ===")

STEP_ORDER = [
    ('raw', 'Raw ChEMBL records (notebook 01)'),
    ('step1_valid_structure', 'Valid, standardised structures'),
    ('step2_endpoint_eligible', 'Endpoint type pooled + unit/activity valid'),
    ('step3_aggregated', 'Aggregated to 1 row per compound'),
    ('step4_missing_dropped', 'Missing essential fields dropped'),
    ('step5_quality_filtered', 'Outside plausible activity range dropped'),
    ('final', 'Final cleaned dataset (steps 6-8 annotate only, no row change)'),
]

attrition_records = []
for tk in TARGETS:
    raw_unique_molecules = df_raw[tk]['molecule_chembl_id'].nunique()
    for pool in POOLS:
        counts = {
            'raw': len(df_raw[tk]),
            'step1_valid_structure': len(df_step1[tk]),
            'step2_endpoint_eligible': len(df_step2[(tk, pool)]),
            'step3_aggregated': len(df_step3[(tk, pool)]),
            'step4_missing_dropped': len(df_step4[(tk, pool)]),
            'step5_quality_filtered': len(df_step5[(tk, pool)]),
            'final': len(df_cleaned[(tk, pool)]),
        }
        prev = None
        for step_key, step_label in STEP_ORDER:
            output_rows = counts[step_key]
            input_rows = counts['raw'] if prev is None else counts[prev]
            attrition_records.append({
                'target': tk, 'activity_pool': pool, 'raw_unique_molecules': raw_unique_molecules,
                'step': step_key, 'step_label': step_label,
                'input_rows': input_rows, 'output_rows': output_rows,
                'removed': input_rows - output_rows,
                'removed_pct': (input_rows - output_rows) / input_rows * 100 if input_rows else 0.0,
            })
            prev = step_key

attrition_df = pd.DataFrame(attrition_records)
print(attrition_df.to_string(index=False))

attrition_path = processed_path / 'cleaning_attrition_by_step.csv'
attrition_df.to_csv(attrition_path, index=False)
_register('cleaning_attrition_by_step.csv', attrition_path, len(attrition_df))
print(f'\nSaved: {attrition_path}')


=== CLEANING ATTRITION LOG (long format, all targets x pools) ===
 target activity_pool  raw_unique_molecules                    step                                                     step_label  input_rows  output_rows  removed  removed_pct
   drd2            ki                 11049                     raw                               Raw ChEMBL records (notebook 01)       17263        17263        0     0.000000
   drd2            ki                 11049   step1_valid_structure                                 Valid, standardised structures       17263        17237       26     0.150611
   drd2            ki                 11049 step2_endpoint_eligible                     Endpoint type pooled + unit/activity valid       17237        13698     3539    20.531415
   drd2            ki                 11049        step3_aggregated                               Aggregated to 1 row per compound       13698         9787     3911    28.551613
   drd2            ki                 11049 

In [35]:
print("=== ENDPOINT COMPOSITION SUMMARY (retained measurements, all targets x pools) ===")
print("(counts here are RETAINED measurements contributing to final compound")
print(" aggregates — not raw ChEMBL record counts from notebook 01, and not")
print(" the endpoint-eligible count before quality filtering.)")

endpoint_records = []
for tk in TARGETS:
    for pool in POOLS:
        df = df_cleaned[(tk, pool)]
        n = len(df)
        n_ki, n_ic50, n_ec50 = df['n_ki'].sum(), df['n_ic50'].sum(), df['n_ec50'].sum()
        total_meas = n_ki + n_ic50 + n_ec50
        n_multi_type = (df['n_activity_types'] > 1).sum()
        endpoint_records.append({
            'target': tk, 'activity_pool': pool,
            'n_retained_ki_measurements': int(n_ki),
            'n_retained_ic50_measurements': int(n_ic50),
            'n_retained_ec50_measurements': int(n_ec50),
            'pct_retained_ki_measurements': n_ki / total_meas * 100 if total_meas else 0.0,
            'pct_retained_ic50_measurements': n_ic50 / total_meas * 100 if total_meas else 0.0,
            'pct_retained_ec50_measurements': n_ec50 / total_meas * 100 if total_meas else 0.0,
            'n_compounds_with_ki': int((df['n_ki'] > 0).sum()),
            'n_compounds_with_ic50': int((df['n_ic50'] > 0).sum()),
            'n_compounds_with_ec50': int((df['n_ec50'] > 0).sum()),
            'n_compounds_multi_endpoint_type': int(n_multi_type),
            'pct_compounds_multi_endpoint_type': n_multi_type / n * 100 if n else 0.0,
        })

endpoint_composition_df = pd.DataFrame(endpoint_records)
print(endpoint_composition_df.to_string(index=False))

endpoint_composition_path = processed_path / 'endpoint_composition_summary.csv'
endpoint_composition_df.to_csv(endpoint_composition_path, index=False)
_register('endpoint_composition_summary.csv', endpoint_composition_path, len(endpoint_composition_df))
print(f'\nSaved: {endpoint_composition_path}')


=== ENDPOINT COMPOSITION SUMMARY (retained measurements, all targets x pools) ===
(counts here are RETAINED measurements contributing to final compound
 aggregates — not raw ChEMBL record counts from notebook 01, and not
 the endpoint-eligible count before quality filtering.)
 target activity_pool  n_retained_ki_measurements  n_retained_ic50_measurements  n_retained_ec50_measurements  pct_retained_ki_measurements  pct_retained_ic50_measurements  pct_retained_ec50_measurements  n_compounds_with_ki  n_compounds_with_ic50  n_compounds_with_ec50  n_compounds_multi_endpoint_type  pct_compounds_multi_endpoint_type
   drd2            ki                       13682                             0                             0                    100.000000                        0.000000                        0.000000                 9775                      0                      0                                0                           0.000000
   drd2       ki_ic50                       1

In [36]:
print("=== ENDPOINT COMBINATION SUMMARY (compound-level, full pool, fixed ordering) ===")

def endpoint_combination(row):
    present = []
    if row['n_ki'] > 0: present.append('Ki')
    if row['n_ic50'] > 0: present.append('IC50')
    if row['n_ec50'] > 0: present.append('EC50')
    return ' + '.join(present) if present else 'none'

endpoint_combo_records = []
for tk in TARGETS:
    df = df_cleaned[(tk, 'full')]
    combos = df.apply(endpoint_combination, axis=1).value_counts()
    for combo, cnt in combos.items():
        endpoint_combo_records.append({
            'target': tk, 'activity_pool': 'full', 'endpoint_combination': combo,
            'compound_count': int(cnt), 'pct_of_dataset': cnt / len(df) * 100,
        })

endpoint_combination_df = pd.DataFrame(endpoint_combo_records)
print(endpoint_combination_df.to_string(index=False))

endpoint_combination_path = processed_path / 'endpoint_combination_summary.csv'
endpoint_combination_df.to_csv(endpoint_combination_path, index=False)
_register('endpoint_combination_summary.csv', endpoint_combination_path, len(endpoint_combination_df))
print(f'\nSaved: {endpoint_combination_path}')


=== ENDPOINT COMBINATION SUMMARY (compound-level, full pool, fixed ordering) ===
 target activity_pool endpoint_combination  compound_count  pct_of_dataset
   drd2          full                   Ki            9030       82.677165
   drd2          full                 IC50             760        6.958433
   drd2          full            Ki + EC50             393        3.598242
   drd2          full            Ki + IC50             260        2.380516
   drd2          full                 EC50             249        2.279802
   drd2          full          IC50 + EC50             137        1.254349
   drd2          full     Ki + IC50 + EC50              93        0.851492
    cb2          full                   Ki            3884       39.192735
    cb2          full                 EC50            3507       35.388496
    cb2          full                 IC50            1098       11.079717
    cb2          full            Ki + EC50             710        7.164480
    cb2          fu

In [37]:
print("=== STRUCTURAL ALERT SUMMARY (PAINS and Brenk kept separate) ===")

alert_records, alert_freq_records = [], []
for tk in TARGETS:
    for pool in POOLS:
        df = df_cleaned[(tk, pool)]
        n = len(df)
        n_pains, n_brenk, n_any = int(df['is_pains'].sum()), int(df['is_brenk'].sum()), int(df['has_any_structural_alert'].sum())
        alert_records.append({
            'target': tk, 'activity_pool': pool,
            'n_pains_flagged': n_pains, 'pains_flagged_pct': n_pains / n * 100 if n else 0.0,
            'n_brenk_flagged': n_brenk, 'brenk_flagged_pct': n_brenk / n * 100 if n else 0.0,
            'n_any_alert': n_any, 'any_alert_pct': n_any / n * 100 if n else 0.0,
        })
        for catalog, reason_col in [('PAINS', 'pains_reason'), ('BRENK', 'brenk_reason')]:
            reasons = df[reason_col].dropna()
            reasons = reasons[reasons != '']
            if reasons.empty:
                continue
            for reason, cnt in reasons.str.split('; ').explode().value_counts().items():
                alert_freq_records.append({
                    'target': tk, 'activity_pool': pool, 'catalogue': catalog,
                    'alert_description': reason, 'compound_count': int(cnt),
                    'pct_of_dataset': cnt / n * 100 if n else 0.0,
                })

structural_alert_summary_df = pd.DataFrame(alert_records)
structural_alert_frequency_df = pd.DataFrame(alert_freq_records)
print(structural_alert_summary_df.to_string(index=False))

structural_alert_summary_path = processed_path / 'structural_alert_summary.csv'
structural_alert_frequency_path = processed_path / 'structural_alert_frequency.csv'
structural_alert_summary_df.to_csv(structural_alert_summary_path, index=False)
structural_alert_frequency_df.to_csv(structural_alert_frequency_path, index=False)
_register('structural_alert_summary.csv', structural_alert_summary_path, len(structural_alert_summary_df))
_register('structural_alert_frequency.csv', structural_alert_frequency_path, len(structural_alert_frequency_df))
print(f'\nSaved: {structural_alert_summary_path}')
print(f'Saved: {structural_alert_frequency_path}')


=== STRUCTURAL ALERT SUMMARY (PAINS and Brenk kept separate) ===
 target activity_pool  n_pains_flagged  pains_flagged_pct  n_brenk_flagged  brenk_flagged_pct  n_any_alert  any_alert_pct
   drd2            ki              326           3.335038             4250          43.478261         4408      45.094629
   drd2       ki_ic50              338           3.167166             4619          43.281484         4785      44.836957
   drd2          full              352           3.222853             4720          43.215528         4888      44.753708
    cb2            ki              180           3.544703             2487          48.975975         2590      51.004332
    cb2       ki_ic50              231           3.608247             2738          42.767885         2862      44.704780
    cb2          full              305           3.077699             3708          37.416751         3860      38.950555
adora2a            ki              216           3.291178             1348       

In [38]:
print("=== ASSAY SUPPORT & MEASUREMENT SUPPORT SUMMARY ===")

MEASUREMENT_BINS = [0, 1, 2, 3, 5, 10, 20, np.inf]
MEASUREMENT_LABELS = ['1', '2', '3', '4-5', '6-10', '11-20', '>20']

assay_support_records, measurement_bins_records = [], []
for tk in TARGETS:
    for pool in POOLS:
        df = df_cleaned[(tk, pool)]
        n = len(df)
        for cat, cnt in df['assay_support'].value_counts().sort_index().items():
            assay_support_records.append({
                'target': tk, 'activity_pool': pool, 'support_category': str(cat),
                'compound_count': int(cnt), 'pct': cnt / n * 100 if n else 0.0,
            })
        meas_binned = pd.cut(df['n_measurements'], bins=MEASUREMENT_BINS, labels=MEASUREMENT_LABELS, include_lowest=True)
        for cat, cnt in meas_binned.value_counts().reindex(MEASUREMENT_LABELS).items():
            measurement_bins_records.append({
                'target': tk, 'activity_pool': pool, 'measurements_bin': str(cat),
                'compound_count': int(cnt), 'pct': cnt / n * 100 if n else 0.0,
            })

assay_support_summary_df = pd.DataFrame(assay_support_records)
measurement_support_distribution_df = pd.DataFrame(measurement_bins_records)
print(assay_support_summary_df.to_string(index=False))

assay_support_summary_path = processed_path / 'assay_support_summary.csv'
measurement_support_distribution_path = processed_path / 'measurement_support_distribution.csv'
assay_support_summary_df.to_csv(assay_support_summary_path, index=False)
measurement_support_distribution_df.to_csv(measurement_support_distribution_path, index=False)
_register('assay_support_summary.csv', assay_support_summary_path, len(assay_support_summary_df))
_register('measurement_support_distribution.csv', measurement_support_distribution_path, len(measurement_support_distribution_df))
print(f'\nSaved: {assay_support_summary_path}')
print(f'Saved: {measurement_support_distribution_path}')


=== ASSAY SUPPORT & MEASUREMENT SUPPORT SUMMARY ===
 target activity_pool support_category  compound_count       pct
   drd2            ki           single            7625 78.005115
   drd2            ki          limited            2019 20.654731
   drd2            ki         moderate             113  1.156010
   drd2            ki           strong              18  0.184143
   drd2       ki_ic50           single            8226 77.080210
   drd2       ki_ic50          limited            2268 21.251874
   drd2       ki_ic50         moderate             159  1.489880
   drd2       ki_ic50           strong              19  0.178036
   drd2          full           single            8088 74.052371
   drd2          full          limited            2538 23.237502
   drd2          full         moderate             272  2.490386
   drd2          full           strong              24  0.219740
    cb2            ki           single            4654 91.650256
    cb2            ki          limited

In [39]:
print("=== PHYSICOCHEMICAL DESCRIPTOR SUMMARY ===")

DESCRIPTOR_COLS = ['MW', 'LogP', 'TPSA', 'HBD', 'HBA', 'RotBonds',
                    'HeavyAtomCount', 'RingCount', 'AromaticRingCount', 'FractionCSP3']

descriptor_records = []
for tk in TARGETS:
    for pool in POOLS:
        df = df_cleaned[(tk, pool)]
        for col in DESCRIPTOR_COLS:
            s = df[col].dropna()
            descriptor_records.append({
                'target': tk, 'activity_pool': pool, 'descriptor': col,
                'count': len(s), 'mean': s.mean(), 'sd': s.std(), 'median': s.median(),
                'q1': s.quantile(0.25), 'q3': s.quantile(0.75), 'iqr': s.quantile(0.75) - s.quantile(0.25),
                'min': s.min(), 'max': s.max(), 'p05': s.quantile(0.05), 'p95': s.quantile(0.95),
            })

physchem_summary_df = pd.DataFrame(descriptor_records)
print(physchem_summary_df.head(20).to_string(index=False))
print(f'... ({len(physchem_summary_df)} rows total: {len(TARGETS)} targets x {len(POOLS)} pools x {len(DESCRIPTOR_COLS)} descriptors)')

physchem_summary_path = processed_path / 'physicochemical_descriptor_summary.csv'
physchem_summary_df.to_csv(physchem_summary_path, index=False)
_register('physicochemical_descriptor_summary.csv', physchem_summary_path, len(physchem_summary_df))
print(f'\nSaved: {physchem_summary_path}')


=== PHYSICOCHEMICAL DESCRIPTOR SUMMARY ===
target activity_pool        descriptor  count       mean         sd     median         q1        q3        iqr       min       max        p05        p95
  drd2            ki                MW   9775 431.259085 165.368953 422.481000 362.864500 477.61100 114.746500 135.16600 3771.2620 269.392000 549.552600
  drd2            ki              LogP   9775   4.059353   1.345920   4.037500   3.268650   4.79705   1.528400 -10.73283   16.7686   2.158720   6.012290
  drd2            ki              TPSA   9775  57.939150  55.443546  54.040000  36.610000  69.31000  32.700000   3.24000 1367.9900  19.030000  93.220000
  drd2            ki               HBD   9775   1.040409   1.921176   1.000000   0.000000   1.00000   1.000000   0.00000   41.0000   0.000000   2.000000
  drd2            ki               HBA   9775   4.599693   2.482404   4.000000   3.000000   6.00000   3.000000   0.00000   58.0000   2.000000   7.000000
  drd2            ki          RotBonds 

In [40]:
print("=== PER-TARGET SCAFFOLD DIVERSITY (extended), ALL TARGETS x ALL POOLS ===")
print("Scaffold IDs computed once here — notebook 03's scaffold split reuses")
print("compound_scaffold_assignments.csv rather than recomputing independently.\n")

import hashlib

N_BOOTSTRAP = 500
_rng = np.random.default_rng(42)
# Bootstrap resamples COMPOUNDS (the actual ChEMBL-curation observational
# unit), then recomputes the scaffold-derived singleton% on each resample —
# standard nonparametric case-resampling. Resampling the scaffold list
# directly instead would treat a 50-compound scaffold and a singleton
# scaffold as equally-weighted draws, which isn't how this data was
# collected. Caveat: these datasets ARE the full curated collections being
# analysed, not a survey sample from a larger population — the CI describes
# resampling uncertainty in the compound-collection process, not sampling
# error from some bigger population.

def _stable_scaffold_id(scaffold_smiles):
    """SHA256-derived ID from the scaffold SMILES itself — the SAME Murcko
    scaffold gets the SAME id across every target/pool it appears in, unlike
    the old frequency-rank id (e.g. 'drd2_full_scaf00001') which was only
    locally meaningful within one (target, pool) and couldn't be used to
    recognise a shared scaffold across pools or targets."""
    digest = hashlib.sha256(scaffold_smiles.encode('utf-8')).hexdigest()[:12]
    return f'SCAF_{digest}'


def scaffold_diversity(df, tk, pool):
    smiles = df['clean_smiles'].tolist()
    chembl_ids = df['molecule_chembl_id'].tolist()
    scaffold_smi = []
    with BlockLogs():
        for smi in smiles:
            mol = Chem.MolFromSmiles(smi)
            if mol:
                try:
                    scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False) or smi
                except Exception:
                    scaf = smi
            else:
                scaf = '__NONE__'
            scaffold_smi.append(scaf)

    assign_df = pd.DataFrame({
        'target': tk, 'activity_pool': pool,
        'clean_smiles': smiles, 'molecule_chembl_id': chembl_ids,
        'murcko_scaffold_smiles': scaffold_smi,
    })
    scaffold_sizes = assign_df.groupby('murcko_scaffold_smiles')['clean_smiles'].transform('size')
    assign_df['scaffold_size'] = scaffold_sizes
    assign_df['is_singleton_scaffold'] = scaffold_sizes == 1
    assign_df['global_scaffold_id'] = assign_df['murcko_scaffold_smiles'].map(_stable_scaffold_id)

    size_by_scaffold = assign_df.groupby('murcko_scaffold_smiles')['clean_smiles'].size().sort_values(ascending=False)
    rank_ids = {scaf: f'{tk}_{pool}_scaf{i:05d}' for i, scaf in enumerate(size_by_scaffold.index)}
    assign_df['within_dataset_scaffold_rank_id'] = assign_df['murcko_scaffold_smiles'].map(rank_ids)

    freq = size_by_scaffold.reset_index(name='compound_count').rename(columns={'murcko_scaffold_smiles': 'scaffold_smiles'})
    freq['within_dataset_scaffold_rank_id'] = freq['scaffold_smiles'].map(rank_ids)
    freq['global_scaffold_id'] = freq['scaffold_smiles'].map(_stable_scaffold_id)
    freq['dataset_pct'] = freq['compound_count'] / len(assign_df) * 100
    freq['rank'] = np.arange(1, len(freq) + 1)
    freq.insert(0, 'activity_pool', pool)
    freq.insert(0, 'target', tk)
    freq = freq[['target', 'activity_pool', 'global_scaffold_id', 'within_dataset_scaffold_rank_id',
                 'scaffold_smiles', 'compound_count', 'dataset_pct', 'rank']]

    n_compounds, n_scaffolds = len(assign_df), freq.shape[0]
    n_singleton = int((freq['compound_count'] == 1).sum())
    singleton_pct = n_singleton / n_scaffolds * 100
    largest_size = int(freq['compound_count'].iloc[0])
    largest_pct = float(freq['dataset_pct'].iloc[0])
    top5_pct = float(freq['compound_count'].iloc[:5].sum() / n_compounds * 100)
    top10_pct = float(freq['compound_count'].iloc[:10].sum() / n_compounds * 100)

    scaffold_arr = assign_df['murcko_scaffold_smiles'].to_numpy()
    boot_pcts = []
    for _ in range(N_BOOTSTRAP):
        sample = _rng.choice(scaffold_arr, size=len(scaffold_arr), replace=True)
        _, counts = np.unique(sample, return_counts=True)
        boot_pcts.append((counts == 1).sum() / len(counts) * 100)
    ci_lo, ci_hi = np.percentile(boot_pcts, [2.5, 97.5])

    stats_row = {
        'target': tk, 'activity_pool': pool,
        'n_compounds': n_compounds, 'n_scaffolds': n_scaffolds,
        'scaffold_richness': n_scaffolds / n_compounds,
        'n_singleton_scaffolds': n_singleton, 'pct_singleton_scaffolds': singleton_pct,
        'singleton_pct_ci_lo': ci_lo, 'singleton_pct_ci_hi': ci_hi,
        'non_singleton_scaffolds': n_scaffolds - n_singleton,
        'pct_non_singleton_scaffolds': 100 - singleton_pct,
        'avg_compounds_per_scaffold': n_compounds / n_scaffolds,
        'median_compounds_per_scaffold': float(freq['compound_count'].median()),
        'scaffold_size_q1': float(freq['compound_count'].quantile(0.25)),
        'scaffold_size_q3': float(freq['compound_count'].quantile(0.75)),
        'scaffold_size_max': largest_size,
        'largest_scaffold_size': largest_size,
        'largest_scaffold_pct': largest_pct,
        'top_5_scaffold_coverage_pct': top5_pct,
        'top_10_scaffold_coverage_pct': top10_pct,
    }

    print(f'{tk.upper()}/{pool}: {n_compounds} compounds -> {n_scaffolds} scaffolds '
          f'(singleton {singleton_pct:.1f}% [{ci_lo:.1f}-{ci_hi:.1f}], largest scaffold {largest_pct:.1f}% of dataset)')

    return assign_df[['target', 'activity_pool', 'clean_smiles', 'molecule_chembl_id',
                       'murcko_scaffold_smiles', 'global_scaffold_id', 'within_dataset_scaffold_rank_id',
                       'scaffold_size', 'is_singleton_scaffold']], freq, stats_row


scaffold_assignment_frames, scaffold_freq_frames, scaffold_summary_records = [], [], []
for tk in TARGETS:
    for pool in POOLS:
        assign_df, freq_df, stats_row = scaffold_diversity(df_cleaned[(tk, pool)], tk, pool)
        scaffold_assignment_frames.append(assign_df)
        scaffold_freq_frames.append(freq_df)
        scaffold_summary_records.append(stats_row)

scaffold_summary_df = pd.DataFrame(scaffold_summary_records)
compound_scaffold_assignments_df = pd.concat(scaffold_assignment_frames, ignore_index=True)
scaffold_frequency_df = pd.concat(scaffold_freq_frames, ignore_index=True)

print()
print('=== Scaffold diversity summary (all targets x pools) ===')
print(scaffold_summary_df.to_string(index=False))


=== PER-TARGET SCAFFOLD DIVERSITY (extended), ALL TARGETS x ALL POOLS ===
Scaffold IDs computed once here — notebook 03's scaffold split reuses
compound_scaffold_assignments.csv rather than recomputing independently.

DRD2/ki: 9775 compounds -> 4071 scaffolds (singleton 66.3% [38.6-41.8], largest scaffold 0.8% of dataset)
DRD2/ki_ic50: 10672 compounds -> 4466 scaffolds (singleton 66.6% [39.0-41.9], largest scaffold 0.8% of dataset)
DRD2/full: 10922 compounds -> 4570 scaffolds (singleton 66.5% [39.1-41.8], largest scaffold 0.9% of dataset)
CB2/ki: 5078 compounds -> 1912 scaffolds (singleton 67.2% [37.8-42.4], largest scaffold 2.4% of dataset)
CB2/ki_ic50: 6402 compounds -> 2472 scaffolds (singleton 67.6% [38.2-42.2], largest scaffold 1.9% of dataset)
CB2/full: 9910 compounds -> 3818 scaffolds (singleton 66.5% [38.3-41.5], largest scaffold 1.3% of dataset)
ADORA2A/ki: 6563 compounds -> 2623 scaffolds (singleton 66.1% [38.2-42.0], largest scaffold 1.3% of dataset)
ADORA2A/ki_ic50: 8171 co

In [41]:
print("=== CROSS-TARGET SCAFFOLD AND COMPOUND OVERLAP (full pool, all target pairs) ===")
print("Descriptive only — five pharmacologically independent targets, not a")
print("selectivity pair, so this is chemical-space overlap bookkeeping (useful")
print("for spotting shared/promiscuous ligands later), not a selectivity analysis.\n")

from itertools import combinations

scaffold_overlap_records, compound_overlap_records = [], []
for target_a, target_b in combinations(TARGETS, 2):
    scaf_a = set(compound_scaffold_assignments_df.query("target == @target_a and activity_pool == 'full'")['global_scaffold_id'])
    scaf_b = set(compound_scaffold_assignments_df.query("target == @target_b and activity_pool == 'full'")['global_scaffold_id'])
    shared_scaf = scaf_a & scaf_b
    union_scaf = scaf_a | scaf_b
    scaffold_overlap_records.append({
        'target_a': target_a, 'target_b': target_b,
        'n_scaffolds_a': len(scaf_a), 'n_scaffolds_b': len(scaf_b),
        'n_shared_scaffolds': len(shared_scaf),
        'jaccard_similarity': len(shared_scaf) / len(union_scaf) if union_scaf else 0.0,
        'pct_a_scaffolds_shared': len(shared_scaf) / len(scaf_a) * 100 if scaf_a else 0.0,
        'pct_b_scaffolds_shared': len(shared_scaf) / len(scaf_b) * 100 if scaf_b else 0.0,
    })

    smiles_a = set(df_cleaned[(target_a, 'full')]['clean_smiles'])
    smiles_b = set(df_cleaned[(target_b, 'full')]['clean_smiles'])
    shared_smiles = smiles_a & smiles_b
    union_smiles = smiles_a | smiles_b
    compound_overlap_records.append({
        'target_a': target_a, 'target_b': target_b,
        'n_compounds_a': len(smiles_a), 'n_compounds_b': len(smiles_b),
        'n_shared_compounds': len(shared_smiles),
        'jaccard_similarity': len(shared_smiles) / len(union_smiles) if union_smiles else 0.0,
        'pct_a_compounds_shared': len(shared_smiles) / len(smiles_a) * 100 if smiles_a else 0.0,
        'pct_b_compounds_shared': len(shared_smiles) / len(smiles_b) * 100 if smiles_b else 0.0,
    })

cross_target_scaffold_overlap_df = pd.DataFrame(scaffold_overlap_records)
cross_target_compound_overlap_df = pd.DataFrame(compound_overlap_records)

print('Scaffold overlap:')
print(cross_target_scaffold_overlap_df.to_string(index=False))
print('\nCompound (canonical SMILES) overlap — flags multi-target/promiscuous ligands:')
print(cross_target_compound_overlap_df.to_string(index=False))

cross_target_scaffold_overlap_path = processed_path / 'cross_target_scaffold_overlap.csv'
cross_target_compound_overlap_path = processed_path / 'cross_target_compound_overlap.csv'
cross_target_scaffold_overlap_df.to_csv(cross_target_scaffold_overlap_path, index=False)
cross_target_compound_overlap_df.to_csv(cross_target_compound_overlap_path, index=False)
_register('cross_target_scaffold_overlap.csv', cross_target_scaffold_overlap_path, len(cross_target_scaffold_overlap_df))
_register('cross_target_compound_overlap.csv', cross_target_compound_overlap_path, len(cross_target_compound_overlap_df))
print(f'\nSaved: {cross_target_scaffold_overlap_path}')
print(f'Saved: {cross_target_compound_overlap_path}')


=== CROSS-TARGET SCAFFOLD AND COMPOUND OVERLAP (full pool, all target pairs) ===
Descriptive only — five pharmacologically independent targets, not a
selectivity pair, so this is chemical-space overlap bookkeeping (useful
for spotting shared/promiscuous ligands later), not a selectivity analysis.

Scaffold overlap:
target_a target_b  n_scaffolds_a  n_scaffolds_b  n_shared_scaffolds  jaccard_similarity  pct_a_scaffolds_shared  pct_b_scaffolds_shared
    drd2      cb2           4570           3818                  40            0.004792                0.875274                1.047669
    drd2  adora2a           4570           3136                  53            0.006925                1.159737                1.690051
    drd2    oprm1           4570           3680                 247            0.030863                5.404814                6.711957
    drd2     ccr5           4570            935                   7            0.001273                0.153173                0.748663
   

In [42]:
scaffold_summary_path = processed_path / 'scaffold_diversity_summary.csv'
compound_scaffold_assignments_path = processed_path / 'compound_scaffold_assignments.csv'
scaffold_frequency_path = processed_path / 'scaffold_frequency_table.csv'

scaffold_summary_df.to_csv(scaffold_summary_path, index=False)
compound_scaffold_assignments_df.to_csv(compound_scaffold_assignments_path, index=False)
scaffold_frequency_df.to_csv(scaffold_frequency_path, index=False)

_register('scaffold_diversity_summary.csv', scaffold_summary_path, len(scaffold_summary_df))
_register('compound_scaffold_assignments.csv', compound_scaffold_assignments_path, len(compound_scaffold_assignments_df))
_register('scaffold_frequency_table.csv', scaffold_frequency_path, len(scaffold_frequency_df))

print(f'Saved: {scaffold_summary_path}')
print(f'Saved: {compound_scaffold_assignments_path} ({len(compound_scaffold_assignments_df)} rows)')
print(f'Saved: {scaffold_frequency_path} ({len(scaffold_frequency_df)} rows)')


Saved: /content/drive/My Drive/gpcr_benchmark/data/processed/scaffold_diversity_summary.csv
Saved: /content/drive/My Drive/gpcr_benchmark/data/processed/compound_scaffold_assignments.csv (104224 rows)
Saved: /content/drive/My Drive/gpcr_benchmark/data/processed/scaffold_frequency_table.csv (41212 rows)


In [43]:
print("=== ACTIVITY POOL INCREMENT ANALYSIS (per target: ki -> ki_ic50 -> full) ===")

increment_records = []
for tk in TARGETS:
    n_ki = len(df_cleaned[(tk, 'ki')])
    n_ki_ic50 = len(df_cleaned[(tk, 'ki_ic50')])
    n_full = len(df_cleaned[(tk, 'full')])

    def _stats(pool):
        df = df_cleaned[(tk, pool)]
        scaf = scaffold_summary_df.query("target == @tk and activity_pool == @pool").iloc[0]
        return {
            'active_pct': (df['activity'] == 1).mean() * 100,
            'pActivity_mean': df['pActivity'].mean(),
            'pActivity_median': df['pActivity'].median(),
            'n_scaffolds': int(scaf['n_scaffolds']),
            'scaffold_richness': scaf['scaffold_richness'],
            'lipinski_pct': df['Lipinski_compliant'].mean() * 100,
            'pains_free_pct': (~df['is_pains']).mean() * 100,
            'brenk_free_pct': (~df['is_brenk']).mean() * 100,
        }

    s_ki, s_ki_ic50, s_full = _stats('ki'), _stats('ki_ic50'), _stats('full')

    increment_records.append({
        'target': tk,
        'ki_compounds': n_ki, 'ki_ic50_compounds': n_ki_ic50, 'full_compounds': n_full,
        'added_by_ic50': n_ki_ic50 - n_ki, 'added_by_ec50': n_full - n_ki_ic50,
        'ic50_increase_pct': (n_ki_ic50 - n_ki) / n_ki * 100 if n_ki else np.nan,
        'ec50_increase_pct': (n_full - n_ki_ic50) / n_ki_ic50 * 100 if n_ki_ic50 else np.nan,
        'active_pct_ki': s_ki['active_pct'], 'active_pct_ki_ic50': s_ki_ic50['active_pct'], 'active_pct_full': s_full['active_pct'],
        'pActivity_mean_ki': s_ki['pActivity_mean'], 'pActivity_mean_ki_ic50': s_ki_ic50['pActivity_mean'], 'pActivity_mean_full': s_full['pActivity_mean'],
        'pActivity_median_ki': s_ki['pActivity_median'], 'pActivity_median_ki_ic50': s_ki_ic50['pActivity_median'], 'pActivity_median_full': s_full['pActivity_median'],
        'n_scaffolds_ki': s_ki['n_scaffolds'], 'n_scaffolds_ki_ic50': s_ki_ic50['n_scaffolds'], 'n_scaffolds_full': s_full['n_scaffolds'],
        'scaffold_richness_ki': s_ki['scaffold_richness'], 'scaffold_richness_ki_ic50': s_ki_ic50['scaffold_richness'], 'scaffold_richness_full': s_full['scaffold_richness'],
        'lipinski_pct_ki': s_ki['lipinski_pct'], 'lipinski_pct_ki_ic50': s_ki_ic50['lipinski_pct'], 'lipinski_pct_full': s_full['lipinski_pct'],
        'pains_free_pct_ki': s_ki['pains_free_pct'], 'pains_free_pct_ki_ic50': s_ki_ic50['pains_free_pct'], 'pains_free_pct_full': s_full['pains_free_pct'],
        'brenk_free_pct_ki': s_ki['brenk_free_pct'], 'brenk_free_pct_ki_ic50': s_ki_ic50['brenk_free_pct'], 'brenk_free_pct_full': s_full['brenk_free_pct'],
    })

activity_pool_increment_df = pd.DataFrame(increment_records)
print(activity_pool_increment_df.to_string(index=False))

activity_pool_increment_path = processed_path / 'activity_pool_increment_summary.csv'
activity_pool_increment_df.to_csv(activity_pool_increment_path, index=False)
_register('activity_pool_increment_summary.csv', activity_pool_increment_path, len(activity_pool_increment_df))
print(f'\nSaved: {activity_pool_increment_path}')


=== ACTIVITY POOL INCREMENT ANALYSIS (per target: ki -> ki_ic50 -> full) ===
 target  ki_compounds  ki_ic50_compounds  full_compounds  added_by_ic50  added_by_ec50  ic50_increase_pct  ec50_increase_pct  active_pct_ki  active_pct_ki_ic50  active_pct_full  pActivity_mean_ki  pActivity_mean_ki_ic50  pActivity_mean_full  pActivity_median_ki  pActivity_median_ki_ic50  pActivity_median_full  n_scaffolds_ki  n_scaffolds_ki_ic50  n_scaffolds_full  scaffold_richness_ki  scaffold_richness_ki_ic50  scaffold_richness_full  lipinski_pct_ki  lipinski_pct_ki_ic50  lipinski_pct_full  pains_free_pct_ki  pains_free_pct_ki_ic50  pains_free_pct_full  brenk_free_pct_ki  brenk_free_pct_ki_ic50  brenk_free_pct_full
   drd2          9775              10672           10922            897            250           9.176471           2.342579      71.764706           69.893178        69.666728           6.561302                6.522464             6.524880                6.480                      6.43           

In [44]:
print("=== CROSS-POOL COMPOUND OVERLAP & NESTING CHECK (per target) ===")

NESTING_VIOLATION_TOLERANCE_PCT = 5.0  # hard-fail only above this; below it, warn.
# ki subset-of ki_ic50 subset-of full is EXPECTED but not a strict mathematical
# guarantee even with fixed (non-IQR) quality bounds: a compound's aggregated
# median pActivity is computed over a DIFFERENT set of measurements per pool
# (ki uses only its Ki rows; ki_ic50 blends in IC50 rows too), so adding
# endpoint types can legitimately shift a compound across the fixed 3-12
# pActivity boundary in either direction for a genuinely heterogeneous
# compound. A handful of such cases is expected data behaviour, not a bug —
# hard-failing on ANY violation would crash all 15 combinations over that. A
# violation rate large enough to exceed the tolerance below, though, signals
# an actual pipeline problem and should stop the run.

overlap_records, nesting_warnings = [], []
for tk in TARGETS:
    ki_set = set(df_cleaned[(tk, 'ki')]['clean_smiles'])
    ki_ic50_set = set(df_cleaned[(tk, 'ki_ic50')]['clean_smiles'])
    full_set = set(df_cleaned[(tk, 'full')]['clean_smiles'])

    overlap_records.append({
        'target': tk,
        'ki_in_ki_ic50_pct': len(ki_set & ki_ic50_set) / len(ki_set) * 100 if ki_set else np.nan,
        'ki_ic50_in_full_pct': len(ki_ic50_set & full_set) / len(ki_ic50_set) * 100 if ki_ic50_set else np.nan,
        'ki_in_full_pct': len(ki_set & full_set) / len(ki_set) * 100 if ki_set else np.nan,
        'new_compounds_from_ic50': len(ki_ic50_set - ki_set),
        'new_compounds_from_ec50': len(full_set - ki_ic50_set),
    })

    ki_violation_pct = len(ki_set - ki_ic50_set) / len(ki_set) * 100 if ki_set else 0.0
    ki_ic50_violation_pct = len(ki_ic50_set - full_set) / len(ki_ic50_set) * 100 if ki_ic50_set else 0.0

    if ki_violation_pct > 0:
        nesting_warnings.append((tk, 'ki_not_in_ki_ic50', len(ki_set - ki_ic50_set), ki_violation_pct))
    if ki_ic50_violation_pct > 0:
        nesting_warnings.append((tk, 'ki_ic50_not_in_full', len(ki_ic50_set - full_set), ki_ic50_violation_pct))

cross_pool_overlap_df = pd.DataFrame(overlap_records)
print(cross_pool_overlap_df.to_string(index=False))

hard_failures = [w for w in nesting_warnings if w[3] > NESTING_VIOLATION_TOLERANCE_PCT]

if nesting_warnings:
    print('\nPool nesting NOT perfectly clean (ki subset-of ki_ic50 subset-of full expected):')
    for tk, kind, n, pct in nesting_warnings:
        print(f'  {tk}: {kind} — {n} compounds ({pct:.1f}%)')
else:
    print('\nPool nesting confirmed: ki subset-of ki_ic50 subset-of full holds exactly for all targets.')

cross_pool_overlap_path = processed_path / 'cross_pool_overlap_summary.csv'
cross_pool_overlap_df.to_csv(cross_pool_overlap_path, index=False)
_register('cross_pool_overlap_summary.csv', cross_pool_overlap_path, len(cross_pool_overlap_df))
print(f'\nSaved: {cross_pool_overlap_path}')

if hard_failures:
    raise AssertionError(
        f'Pool-nesting violation exceeds {NESTING_VIOLATION_TOLERANCE_PCT}% tolerance for: '
        f'{[(tk, kind, f"{pct:.1f}%") for tk, kind, n, pct in hard_failures]} — '
        f'this is large enough to indicate an actual pipeline problem, not expected edge-case behaviour.'
    )


=== CROSS-POOL COMPOUND OVERLAP & NESTING CHECK (per target) ===
 target  ki_in_ki_ic50_pct  ki_ic50_in_full_pct  ki_in_full_pct  new_compounds_from_ic50  new_compounds_from_ec50
   drd2         100.000000                100.0      100.000000                      897                      250
    cb2         100.000000                100.0      100.000000                     1324                     3508
adora2a          99.984763                100.0       99.984763                     1609                      146
  oprm1         100.000000                100.0      100.000000                     1545                     1468
   ccr5         100.000000                100.0      100.000000                     2238                       39

Pool nesting NOT perfectly clean (ki subset-of ki_ic50 subset-of full expected):
  adora2a: ki_not_in_ki_ic50 — 1 compounds (0.0%)

Saved: /content/drive/My Drive/gpcr_benchmark/data/processed/cross_pool_overlap_summary.csv


In [45]:
print("=== MASTER DATASET SUMMARY (all targets x pools) — authoritative machine-readable table ===")

TARGET_CHEMBL_IDS = {'drd2': 'CHEMBL217', 'cb2': 'CHEMBL253', 'adora2a': 'CHEMBL251', 'oprm1': 'CHEMBL233', 'ccr5': 'CHEMBL274'}

master_records = []
for tk in TARGETS:
    raw_unique_molecules = df_raw[tk]['molecule_chembl_id'].nunique()
    for pool in POOLS:
        df = df_cleaned[(tk, pool)]
        n = len(df)
        att = attrition_df.query("target == @tk and activity_pool == @pool").set_index('step')
        n_active = int((df['activity'] == 1).sum())
        n_inactive = n - n_active
        pa = df['pActivity']

        row = {
            'target': tk, 'target_chembl_id': TARGET_CHEMBL_IDS[tk], 'activity_pool': pool,
            'activity_types': ', '.join(POOL_TYPES[pool]),
            'raw_records': int(att.loc['raw', 'output_rows']),
            'raw_unique_molecules': int(raw_unique_molecules),
            'endpoint_eligible_records': int(att.loc['step2_endpoint_eligible', 'output_rows']),
            'final_unique_compounds': n,
            'molecule_retention_pct': n / raw_unique_molecules * 100,
            'record_to_compound_yield_pct': n / int(att.loc['raw', 'output_rows']) * 100,

            'active_threshold': ACTIVITY_THRESHOLD,
            'n_active': n_active, 'n_inactive': n_inactive,
            'active_pct': n_active / n * 100 if n else 0.0, 'inactive_pct': n_inactive / n * 100 if n else 0.0,
            'pActivity_mean': pa.mean(), 'pActivity_sd': pa.std(), 'pActivity_median': pa.median(),
            'pActivity_q1': pa.quantile(0.25), 'pActivity_q3': pa.quantile(0.75),
            'pActivity_iqr': pa.quantile(0.75) - pa.quantile(0.25),
            'pActivity_min': pa.min(), 'pActivity_max': pa.max(),
            'n_iqr_outliers': int(df['pActivity_iqr_outlier'].sum()),
            'iqr_outlier_pct': df['pActivity_iqr_outlier'].mean() * 100,

            'measurements_mean': df['n_measurements'].mean(), 'measurements_median': df['n_measurements'].median(),
            'measurements_max': df['n_measurements'].max(),
            'assays_mean': df['n_assays'].mean(), 'assays_median': df['n_assays'].median(), 'assays_max': df['n_assays'].max(),
            'single_measurement_pct': (df['n_measurements'] == 1).mean() * 100,
            'single_assay_pct': (df['n_assays'] == 1).mean() * 100,
            'multi_activity_type_pct': (df['n_activity_types'] > 1).mean() * 100,

            'lipinski_compliant_pct': df['Lipinski_compliant'].mean() * 100,
            'soft_druglike_pct': df['DrugLike_soft'].mean() * 100,
            'pains_free_pct': (~df['is_pains']).mean() * 100,
            'brenk_free_pct': (~df['is_brenk']).mean() * 100,
            'any_alert_free_pct': (~df['has_any_structural_alert']).mean() * 100,
        }

        scaf_row = scaffold_summary_df.query("target == @tk and activity_pool == @pool").iloc[0]
        row.update({
            'n_scaffolds': int(scaf_row['n_scaffolds']),
            'scaffold_richness': scaf_row['scaffold_richness'],
            'singleton_scaffold_pct': scaf_row['pct_singleton_scaffolds'],
            'largest_scaffold_size': int(scaf_row['largest_scaffold_size']),
            'largest_scaffold_pct': scaf_row['largest_scaffold_pct'],
            'median_scaffold_size': scaf_row['median_compounds_per_scaffold'],
        })

        master_records.append(row)

dataset_summary_df = pd.DataFrame(master_records)
print(dataset_summary_df.to_string(index=False))

dataset_summary_path = processed_path / 'dataset_summary_all_targets_pools.csv'
dataset_summary_df.to_csv(dataset_summary_path, index=False)
_register('dataset_summary_all_targets_pools.csv', dataset_summary_path, len(dataset_summary_df))
print(f'\nSaved: {dataset_summary_path}')


=== MASTER DATASET SUMMARY (all targets x pools) — authoritative machine-readable table ===
 target target_chembl_id activity_pool activity_types  raw_records  raw_unique_molecules  endpoint_eligible_records  final_unique_compounds  molecule_retention_pct  record_to_compound_yield_pct  active_threshold  n_active  n_inactive  active_pct  inactive_pct  pActivity_mean  pActivity_sd  pActivity_median  pActivity_q1  pActivity_q3  pActivity_iqr  pActivity_min  pActivity_max  n_iqr_outliers  iqr_outlier_pct  measurements_mean  measurements_median  measurements_max  assays_mean  assays_median  assays_max  single_measurement_pct  single_assay_pct  multi_activity_type_pct  lipinski_compliant_pct  soft_druglike_pct  pains_free_pct  brenk_free_pct  any_alert_free_pct  n_scaffolds  scaffold_richness  singleton_scaffold_pct  largest_scaffold_size  largest_scaffold_pct  median_scaffold_size
   drd2        CHEMBL217            ki             Ki        17263                 11049                      1

In [46]:
print("=== PACTIVITY IQR-OUTLIER FLAG SUMMARY (annotation only, not an exclusion) ===")

outlier_records = []
for tk in TARGETS:
    for pool in POOLS:
        df = df_cleaned[(tk, pool)]
        pa = df['pActivity']
        q1, q3 = pa.quantile(0.25), pa.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_flagged = int(df['pActivity_iqr_outlier'].sum())
        outlier_records.append({
            'target': tk, 'activity_pool': pool,
            'Q1': q1, 'Q3': q3, 'IQR': iqr,
            'lower_flag_boundary': lower, 'upper_flag_boundary': upper,
            'n_flagged_compounds': n_flagged,
            'flagged_pct': n_flagged / len(df) * 100 if len(df) else 0.0,
        })

pactivity_outlier_df = pd.DataFrame(outlier_records)
print(pactivity_outlier_df.to_string(index=False))

pactivity_outlier_path = processed_path / 'pactivity_outlier_summary.csv'
pactivity_outlier_df.to_csv(pactivity_outlier_path, index=False)
_register('pactivity_outlier_summary.csv', pactivity_outlier_path, len(pactivity_outlier_df))
print(f'\nSaved: {pactivity_outlier_path}')


=== PACTIVITY IQR-OUTLIER FLAG SUMMARY (annotation only, not an exclusion) ===
 target activity_pool       Q1     Q3      IQR  lower_flag_boundary  upper_flag_boundary  n_flagged_compounds  flagged_pct
   drd2            ki 5.890000 7.2325 1.342500             3.876250             9.246250                  113     1.156010
   drd2       ki_ic50 5.820000 7.2300 1.410000             3.705000             9.345000                   89     0.833958
   drd2          full 5.810000 7.2500 1.440000             3.650000             9.410000                   73     0.668376
    cb2            ki 5.680000 7.7000 2.020000             2.650000            10.730000                    0     0.000000
    cb2       ki_ic50 5.460000 7.5700 2.110000             2.295000            10.735000                    0     0.000000
    cb2          full 5.522879 7.6500 2.127121             2.332197            10.840682                    0     0.000000
adora2a            ki 5.820000 7.8400 2.020000             2

## Dataset Characterisation: Figures

10 publication-ready figures (300dpi PNG + vector PDF), each with its
plot-source data saved alongside — never just the rendered image. Main
figures below use the `full` pool for clarity where a 15-way facet would be
unreadable; the underlying per-pool data is already saved in the tables
above for supplementary use.

In [47]:
print("=== DATASET ATTRITION (full pool, all targets) ===")

attrition_plot_df = attrition_df[attrition_df['activity_pool'] == 'full'].copy()
STEP_LABELS_SHORT = ['Raw', 'Valid\nstructure', 'Endpoint\neligible', 'Aggregated\n(compounds)', 'Final']
STEP_KEYS_PLOT = ['raw', 'step1_valid_structure', 'step2_endpoint_eligible', 'step3_aggregated', 'final']

fig, ax = plt.subplots(figsize=(8, 5))
for tk in TARGETS:
    sub = attrition_plot_df[attrition_plot_df['target'] == tk].set_index('step')
    values = [sub.loc[k, 'output_rows'] for k in STEP_KEYS_PLOT]
    ax.plot(range(len(values)), values, marker='o', label=tk.upper(), color=TARGET_COLORS[tk], linewidth=2)

ax.axvline(2.5, color=GREY, linestyle='--', linewidth=1, zorder=0)
ax.text(2.5, ax.get_ylim()[1], ' activity records -> unique compounds', va='bottom', ha='center', fontsize=7, color=GREY)
ax.set_xticks(range(len(STEP_LABELS_SHORT)))
ax.set_xticklabels(STEP_LABELS_SHORT)
ax.set_ylabel('Count (activity records before aggregation,\nunique compounds after)')
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
save_fig(fig, 'figure_dataset_attrition')

attrition_data_path = processed_path / 'figure_dataset_attrition_data.csv'
attrition_plot_df.to_csv(attrition_data_path, index=False)
register_figure('figure_dataset_attrition')
_register('figure_dataset_attrition_data.csv', attrition_data_path, len(attrition_plot_df))


=== DATASET ATTRITION (full pool, all targets) ===
Figure saved: /content/drive/My Drive/gpcr_benchmark/data/processed/figures/figure_dataset_attrition.png / .pdf


In [48]:
print("=== ACTIVITY POOL EXPANSION ===")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

x = np.arange(len(TARGETS))
width = 0.25
for i, pool in enumerate(POOLS):
    vals = [len(df_cleaned[(tk, pool)]) for tk in TARGETS]
    axes[0].bar(x + (i - 1) * width, vals, width, label=POOL_LABELS[pool], color=[BLUE, ORANGE, RED][i])
axes[0].set_xticks(x); axes[0].set_xticklabels([tk.upper() for tk in TARGETS])
axes[0].set_ylabel('Final compounds')
panel_label(axes[0], 'A')
axes[0].legend(frameon=False)

ic50_inc = activity_pool_increment_df.set_index('target')['ic50_increase_pct']
ec50_inc = activity_pool_increment_df.set_index('target')['ec50_increase_pct']
axes[1].bar(x - 0.2, [ic50_inc[tk] for tk in TARGETS], 0.4, label='Ki to Ki+IC50 (adding IC50)', color=ORANGE)
axes[1].bar(x + 0.2, [ec50_inc[tk] for tk in TARGETS], 0.4, label='Ki+IC50 to full (adding EC50)', color=RED)
axes[1].set_xticks(x); axes[1].set_xticklabels([tk.upper() for tk in TARGETS])
axes[1].set_ylabel('% increase in compounds')
panel_label(axes[1], 'B')
axes[1].legend(frameon=False)
axes[1].axhline(0, color='black', linewidth=0.8)

fig.tight_layout()
save_fig(fig, 'figure_pool_expansion')
register_figure('figure_pool_expansion')
print('Plot-source data: activity_pool_increment_summary.csv (already saved)')


=== ACTIVITY POOL EXPANSION ===
Figure saved: /content/drive/My Drive/gpcr_benchmark/data/processed/figures/figure_pool_expansion.png / .pdf
Plot-source data: activity_pool_increment_summary.csv (already saved)


In [49]:
print("=== ACTIVITY CLASS BALANCE (all 15 datasets) ===")

fig, ax = plt.subplots(figsize=(10, 5))
labels = [f'{tk.upper()}\n{POOL_LABELS[pool]}' for tk in TARGETS for pool in POOLS]
active_pct = [dataset_summary_df.query("target==@tk and activity_pool==@pool")['active_pct'].iloc[0] for tk in TARGETS for pool in POOLS]
inactive_pct = [100 - v for v in active_pct]

x = np.arange(len(labels))
ax.bar(x, active_pct, label='Active', color=BLUE)
ax.bar(x, inactive_pct, bottom=active_pct, label='Inactive', color=GREY)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8, rotation=45, ha='right')
ax.set_ylabel('% of dataset')
ax.legend(frameon=False)
ax.axhline(100, color='black', linewidth=0.5)
fig.tight_layout()
save_fig(fig, 'figure_class_balance')

class_balance_data_path = processed_path / 'figure_class_balance_data.csv'
dataset_summary_df[['target', 'activity_pool', 'n_active', 'n_inactive', 'active_pct', 'inactive_pct']].to_csv(
    class_balance_data_path, index=False)
register_figure('figure_class_balance')
_register('figure_class_balance_data.csv', class_balance_data_path, len(dataset_summary_df))


=== ACTIVITY CLASS BALANCE (all 15 datasets) ===
Figure saved: /content/drive/My Drive/gpcr_benchmark/data/processed/figures/figure_class_balance.png / .pdf


In [50]:
print("=== PACTIVITY DISTRIBUTIONS ===")

fig, ax = plt.subplots(figsize=(10, 5))
plot_data, positions, tick_labels, colors = [], [], [], []
pos = 0
for tk in TARGETS:
    for pool in POOLS:
        plot_data.append(df_cleaned[(tk, pool)]['pActivity'].values)
        positions.append(pos); tick_labels.append(f'{tk.upper()}\n{POOL_LABELS[pool]}'); colors.append(TARGET_COLORS[tk])
        pos += 1
    pos += 0.5

parts = ax.violinplot(plot_data, positions=positions, showmedians=True, widths=0.8)
for pc, c in zip(parts['bodies'], colors):
    pc.set_facecolor(c); pc.set_alpha(0.6)
ax.axhline(ACTIVITY_THRESHOLD, color=RED, linestyle='--', linewidth=1, label=f'Active threshold ({ACTIVITY_THRESHOLD})')
ax.set_xticks(positions); ax.set_xticklabels(tick_labels, fontsize=8, rotation=45, ha='right')
ax.set_ylabel('pActivity')
ax.legend(frameon=False)
fig.tight_layout()
save_fig(fig, 'figure_pactivity_distributions')

pactivity_plot_records = []
for tk in TARGETS:
    for pool in POOLS:
        for v in df_cleaned[(tk, pool)]['pActivity'].values:
            pactivity_plot_records.append({'target': tk, 'activity_pool': pool, 'pActivity': v})
pactivity_data_df = pd.DataFrame(pactivity_plot_records)
pactivity_data_path = processed_path / 'pactivity_distribution_plot_data.csv'
pactivity_data_df.to_csv(pactivity_data_path, index=False)
register_figure('figure_pactivity_distributions')
_register('pactivity_distribution_plot_data.csv', pactivity_data_path, len(pactivity_data_df))


=== PACTIVITY DISTRIBUTIONS ===
Figure saved: /content/drive/My Drive/gpcr_benchmark/data/processed/figures/figure_pactivity_distributions.png / .pdf


In [51]:
print("=== ENDPOINT COMPOSITION (figure) ===")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

full_ep = endpoint_composition_df[endpoint_composition_df['activity_pool'] == 'full'].set_index('target')
x = np.arange(len(TARGETS))
bottom = np.zeros(len(TARGETS))
for typ, color in [('pct_retained_ki_measurements', BLUE), ('pct_retained_ic50_measurements', ORANGE), ('pct_retained_ec50_measurements', RED)]:
    vals = np.array([full_ep.loc[tk, typ] for tk in TARGETS])
    label = typ.replace('pct_retained_', '').replace('_measurements', '').upper()
    axes[0].bar(x, vals, bottom=bottom, label=label, color=color)
    bottom += vals
axes[0].set_xticks(x); axes[0].set_xticklabels([tk.upper() for tk in TARGETS])
axes[0].set_ylabel('% of retained measurements (full pool)')
panel_label(axes[0], 'A')
axes[0].legend(frameon=False)

# Reuses endpoint_combination_summary.csv (fixed Ki/IC50/EC50 ordering) computed above
# rather than recomputing with an alphabetically-sorted combo label.
ENDPOINT_COMBO_ORDER = ['Ki', 'IC50', 'EC50', 'Ki + IC50', 'Ki + EC50', 'IC50 + EC50', 'Ki + IC50 + EC50', 'none']
combo_categories = [c for c in ENDPOINT_COMBO_ORDER if c in endpoint_combination_df['endpoint_combination'].unique()]
combo_colors = dict(zip(combo_categories, plt.cm.tab10.colors))
bottom = np.zeros(len(TARGETS))
for combo in combo_categories:
    vals = np.array([
        endpoint_combination_df.query("target==@tk and endpoint_combination==@combo")['pct_of_dataset'].sum()
        for tk in TARGETS
    ])
    axes[1].bar(x, vals, bottom=bottom, label=combo, color=combo_colors[combo])
    bottom += vals
axes[1].set_xticks(x); axes[1].set_xticklabels([tk.upper() for tk in TARGETS])
axes[1].set_ylabel('% of compounds (full pool)')
panel_label(axes[1], 'B')
axes[1].legend(frameon=False, fontsize=7, ncol=2)

fig.tight_layout()
save_fig(fig, 'figure_endpoint_composition')
register_figure('figure_endpoint_composition')
print('Plot-source data: endpoint_composition_summary.csv, endpoint_combination_summary.csv (already saved)')


=== ENDPOINT COMPOSITION (figure) ===
Figure saved: /content/drive/My Drive/gpcr_benchmark/data/processed/figures/figure_endpoint_composition.png / .pdf
Plot-source data: endpoint_composition_summary.csv, endpoint_combination_summary.csv (already saved)


In [53]:
print("=== SCAFFOLD DIVERSITY ACROSS TARGETS (full pool) ===")

scaf_full = scaffold_summary_df[scaffold_summary_df['activity_pool'] == 'full'].set_index('target').loc[TARGETS]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

x = np.arange(len(TARGETS))
axes[0].bar(x - 0.2, scaf_full['n_compounds'], 0.4, label='Compounds', color=BLUE)
axes[0].bar(x + 0.2, scaf_full['n_scaffolds'], 0.4, label='Scaffolds', color=ORANGE)
axes[0].set_xticks(x); axes[0].set_xticklabels([tk.upper() for tk in TARGETS])
axes[0].set_ylabel('Count')
panel_label(axes[0], 'A')
axes[0].legend(frameon=False)

yerr = np.array([
    np.clip(scaf_full['pct_singleton_scaffolds'] - scaf_full['singleton_pct_ci_lo'], 0, None),
    np.clip(scaf_full['singleton_pct_ci_hi'] - scaf_full['pct_singleton_scaffolds'], 0, None),
])
axes[1].bar(x, scaf_full['pct_singleton_scaffolds'], color=[TARGET_COLORS[tk] for tk in TARGETS], yerr=yerr, capsize=4)
axes[1].set_xticks(x); axes[1].set_xticklabels([tk.upper() for tk in TARGETS])
axes[1].set_ylabel('Singleton scaffolds (%)')
panel_label(axes[1], 'B')

axes[2].bar(x - 0.2, scaf_full['largest_scaffold_pct'], 0.2, label='Largest scaffold', color=RED)
axes[2].bar(x, scaf_full['top_5_scaffold_coverage_pct'], 0.2, label='Top 5 scaffolds', color=ORANGE)
axes[2].bar(x + 0.2, scaf_full['top_10_scaffold_coverage_pct'], 0.2, label='Top 10 scaffolds', color=BLUE)
axes[2].set_xticks(x); axes[2].set_xticklabels([tk.upper() for tk in TARGETS])
axes[2].set_ylabel('% of dataset')
panel_label(axes[2], 'C')
axes[2].legend(frameon=False, fontsize=8)

fig.tight_layout()
save_fig(fig, 'figure_scaffold_diversity')
register_figure('figure_scaffold_diversity')
print('Plot-source data: scaffold_diversity_summary.csv (already saved, all pools)')


=== SCAFFOLD DIVERSITY ACROSS TARGETS (full pool) ===
Figure saved: /content/drive/My Drive/gpcr_benchmark/data/processed/figures/figure_scaffold_diversity.png / .pdf
Plot-source data: scaffold_diversity_summary.csv (already saved, all pools)


In [54]:
print("=== MOLECULAR PROPERTY DISTRIBUTIONS (full pool) ===")

props_to_plot = ['MW', 'LogP', 'TPSA', 'RotBonds', 'FractionCSP3', 'AromaticRingCount']
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, prop in zip(axes.flat, props_to_plot):
    data = [df_cleaned[(tk, 'full')][prop].dropna().values for tk in TARGETS]
    bp = ax.boxplot(data, labels=[tk.upper() for tk in TARGETS], patch_artist=True, showfliers=False)
    for patch, tk in zip(bp['boxes'], TARGETS):
        patch.set_facecolor(TARGET_COLORS[tk]); patch.set_alpha(0.7)
    ax.set_title(DESCRIPTOR_LABELS.get(prop, prop))
    ax.tick_params(axis='x', rotation=45)

fig.tight_layout()
save_fig(fig, 'figure_property_distributions')
register_figure('figure_property_distributions')
print('Plot-source data: compound-level values from cleaned_data_{target}_full.csv')
print('(physicochemical_descriptor_summary.csv has the aggregate stats, not the raw')
print(' per-compound values a boxplot needs — those are already in the 15 cleaned CSVs)')


=== MOLECULAR PROPERTY DISTRIBUTIONS (full pool) ===
Figure saved: /content/drive/My Drive/gpcr_benchmark/data/processed/figures/figure_property_distributions.png / .pdf
Plot-source data: compound-level values from cleaned_data_{target}_full.csv
(physicochemical_descriptor_summary.csv has the aggregate stats, not the raw
 per-compound values a boxplot needs — those are already in the 15 cleaned CSVs)


In [55]:
print("=== CHEMICAL SPACE PCA (full pool, all targets) ===")
print("Note: points are target-compound observations, not independent compounds")
print("— a canonical compound present in more than one target's dataset appears")
print("more than once (once per target it was tested against). global_compound_id")
print("(SHA256 of clean_smiles) is included so such duplicates can be recognised.\n")

import hashlib
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pca_cols = ['MW', 'LogP', 'TPSA', 'HBD', 'HBA', 'RotBonds', 'HeavyAtomCount', 'RingCount', 'AromaticRingCount', 'FractionCSP3']

pca_input_frames = []
for tk in TARGETS:
    df = df_cleaned[(tk, 'full')][['clean_smiles'] + pca_cols].dropna().copy()
    df['target'] = tk
    pca_input_frames.append(df)
pca_input_df = pd.concat(pca_input_frames, ignore_index=True)
pca_input_df['global_compound_id'] = pca_input_df['clean_smiles'].apply(
    lambda s: 'CMPD_' + hashlib.sha256(s.encode('utf-8')).hexdigest()[:12])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_input_df[pca_cols])
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
pca_input_df['PC1'] = coords[:, 0]
pca_input_df['PC2'] = coords[:, 1]

loadings_df = pd.DataFrame(pca.components_.T, index=pca_cols, columns=['PC1', 'PC2']).reset_index().rename(columns={'index': 'descriptor'})
variance_df = pd.DataFrame({'component': ['PC1', 'PC2'], 'explained_variance_ratio': pca.explained_variance_ratio_})

# Equal-sample visualisation only (full coordinates for every compound saved regardless).
# pd.concat + per-target .sample() instead of groupby(...).apply(...) — the
# latter raises a pandas DeprecationWarning about grouping-column handling.
min_n = int(pca_input_df['target'].value_counts().min())
plot_sample = pd.concat(
    [pca_input_df[pca_input_df['target'] == tk].sample(n=min_n, random_state=42) for tk in TARGETS],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(7, 6))
for tk in TARGETS:
    sub = plot_sample[plot_sample['target'] == tk]
    ax.scatter(sub['PC1'], sub['PC2'], s=8, alpha=0.4, color=TARGET_COLORS[tk], label=tk.upper())
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.legend(frameon=False, markerscale=2)
fig.tight_layout()
save_fig(fig, 'figure_chemical_space_pca')

pca_coords_path = processed_path / 'chemical_space_pca_coordinates.csv'
pca_loadings_path = processed_path / 'chemical_space_pca_loadings.csv'
pca_variance_path = processed_path / 'chemical_space_pca_variance.csv'
pca_input_df.to_csv(pca_coords_path, index=False)
loadings_df.to_csv(pca_loadings_path, index=False)
variance_df.to_csv(pca_variance_path, index=False)
register_figure('figure_chemical_space_pca')
_register('chemical_space_pca_coordinates.csv', pca_coords_path, len(pca_input_df))
_register('chemical_space_pca_loadings.csv', pca_loadings_path, len(loadings_df))
_register('chemical_space_pca_variance.csv', pca_variance_path, len(variance_df))
print('Saved: chemical_space_pca_coordinates.csv (all compounds), _loadings.csv, _variance.csv')


=== CHEMICAL SPACE PCA (full pool, all targets) ===
Note: points are target-compound observations, not independent compounds
— a canonical compound present in more than one target's dataset appears
more than once (once per target it was tested against). global_compound_id
(SHA256 of clean_smiles) is included so such duplicates can be recognised.

Figure saved: /content/drive/My Drive/gpcr_benchmark/data/processed/figures/figure_chemical_space_pca.png / .pdf
Saved: chemical_space_pca_coordinates.csv (all compounds), _loadings.csv, _variance.csv


In [56]:
print("=== MEASUREMENT & ASSAY SUPPORT ===")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for tk in TARGETS:
    vals = df_cleaned[(tk, 'full')]['n_measurements']
    axes[0].hist(vals, bins=np.logspace(0, np.log10(max(vals.max(), 2)), 20), alpha=0.5, label=tk.upper(), color=TARGET_COLORS[tk])
axes[0].set_xscale('log')
axes[0].set_xlabel('Measurements per compound (log scale)')
axes[0].set_ylabel('Compound count')
panel_label(axes[0], 'A')
axes[0].legend(frameon=False, fontsize=8)

support_full = assay_support_summary_df[assay_support_summary_df['activity_pool'] == 'full']
categories = ['single', 'limited', 'moderate', 'strong']
x = np.arange(len(TARGETS))
bottom = np.zeros(len(TARGETS))
for cat, color in zip(categories, [RED, ORANGE, BLUE, '#2ca02c']):
    vals = np.array([support_full.query("target==@tk and support_category==@cat")['pct'].sum() for tk in TARGETS])
    axes[1].bar(x, vals, bottom=bottom, label=cat, color=color)
    bottom += vals
axes[1].set_xticks(x); axes[1].set_xticklabels([tk.upper() for tk in TARGETS])
axes[1].set_ylabel('% of compounds')
panel_label(axes[1], 'B')
axes[1].legend(frameon=False, fontsize=8)

fig.tight_layout()
save_fig(fig, 'figure_assay_support')
register_figure('figure_assay_support')
print('Plot-source data: assay_support_summary.csv (panel B, already saved); panel A')
print('(histogram) uses per-compound n_measurements directly from cleaned_data_{target}_full.csv')


=== MEASUREMENT & ASSAY SUPPORT ===
Figure saved: /content/drive/My Drive/gpcr_benchmark/data/processed/figures/figure_assay_support.png / .pdf
Plot-source data: assay_support_summary.csv (panel B, already saved); panel A
(histogram) uses per-compound n_measurements directly from cleaned_data_{target}_full.csv


In [57]:
print("=== STRUCTURAL ALERTS (PAINS vs Brenk, separated) ===")

alert_full = structural_alert_summary_df[structural_alert_summary_df['activity_pool'] == 'full'].set_index('target').loc[TARGETS]

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(TARGETS))
width = 0.25
ax.bar(x - width, alert_full['pains_flagged_pct'], width, label='PAINS', color=RED)
ax.bar(x, alert_full['brenk_flagged_pct'], width, label='Brenk', color=ORANGE)
ax.bar(x + width, alert_full['any_alert_pct'], width, label='Any alert', color=GREY)
ax.set_xticks(x); ax.set_xticklabels([tk.upper() for tk in TARGETS])
ax.set_ylabel('% of compounds flagged')
ax.legend(frameon=False)
fig.tight_layout()
save_fig(fig, 'figure_structural_alerts')

struct_alert_fig_data_path = processed_path / 'figure_structural_alerts_data.csv'
structural_alert_summary_df.to_csv(struct_alert_fig_data_path, index=False)
register_figure('figure_structural_alerts')
_register('figure_structural_alerts_data.csv', struct_alert_fig_data_path, len(structural_alert_summary_df))


=== STRUCTURAL ALERTS (PAINS vs Brenk, separated) ===
Figure saved: /content/drive/My Drive/gpcr_benchmark/data/processed/figures/figure_structural_alerts.png / .pdf


In [58]:
FULL_CONFIG = {
    'targets': TARGETS,
    'pools': POOLS,
    'pool_types': POOL_TYPES,
    'activity_threshold': ACTIVITY_THRESHOLD,
    'concentration_range_nm': [0.1, 1_000_000],
    'pActivity_range': [3.0, 12.0],
    'iqr_filtering': 'non-destructive flag only (pActivity_iqr_outlier) — fixed bounds above are the actual filter',
    'structure_standardisation_sequence': ['Cleanup', 'FragmentParent', 'Normalizer.normalize', 'Uncharger.uncharge', 'TautomerEnumerator.Canonicalize'],
    'aggregation_statistic': 'median (per compound, across replicate measurements and pooled activity types)',
    'structural_alert_catalogues': ['PAINS', 'BRENK'],  # kept as separate flags, not merged
    'min_assays_filter': 1,
    'rdkit_version': rdkit.__version__,
    'bootstrap_resamples_scaffold_ci': N_BOOTSTRAP,
    'nesting_violation_tolerance_pct': NESTING_VIOLATION_TOLERANCE_PCT,
    'random_seed': 42,
}

# ---- Input provenance: hash notebook 01's raw pulls + its manifest (best-effort chain) ----
manifest_inputs = {}
for tk in TARGETS:
    raw_path = base_path / f'raw_data_{tk}.csv'
    manifest_inputs[f'raw_data_{tk}.csv'] = {
        'path': str(raw_path),
        'n_rows': len(df_raw[tk]),
        'n_unique_molecules': int(df_raw[tk]['molecule_chembl_id'].nunique()),
    }

nb01_manifest_path = base_path / 'manifest_01_data_collection.json'
if nb01_manifest_path.exists():
    manifest_inputs['manifest_01_data_collection.json'] = {'path': str(nb01_manifest_path)}

# Optional: hash this notebook's own file for full provenance (set the path if
# known — Colab/Jupyter notebooks have no reliable self-path at runtime, so
# this is skipped gracefully rather than guessed).
NOTEBOOK_PATH = None  # e.g. Path('/content/drive/My Drive/gpcr_benchmark/02_data_cleaning.ipynb')
if NOTEBOOK_PATH is not None and Path(NOTEBOOK_PATH).exists():
    manifest_inputs['02_data_cleaning.ipynb'] = {'path': str(NOTEBOOK_PATH)}

write_manifest(
    processed_path / 'manifest_02_data_cleaning.json',
    config_summary=FULL_CONFIG,
    outputs={
        **{output_filename[(tk, pool)]: {
               'path': str(processed_path / output_filename[(tk, pool)]),
               'n_rows': len(df_save[(tk, pool)]),
           }
           for tk in TARGETS for pool in POOLS},
        **all_outputs,
    },
    inputs=manifest_inputs,
)


Manifest saved: /content/drive/My Drive/gpcr_benchmark/data/processed/manifest_02_data_cleaning.json (6 inputs, 60 outputs hashed)


{'timestamp': '2026-07-24T20:41:53.181779',
 'python_version': '3.12.13',
 'git_commit': None,
 'config': {'targets': ['drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5'],
  'pools': ['ki', 'ki_ic50', 'full'],
  'pool_types': {'ki': ['Ki'],
   'ki_ic50': ['Ki', 'IC50'],
   'full': ['Ki', 'IC50', 'EC50']},
  'activity_threshold': 6.0,
  'concentration_range_nm': [0.1, 1000000],
  'pActivity_range': [3.0, 12.0],
  'iqr_filtering': 'non-destructive flag only (pActivity_iqr_outlier) — fixed bounds above are the actual filter',
  'structure_standardisation_sequence': ['Cleanup',
   'FragmentParent',
   'Normalizer.normalize',
   'Uncharger.uncharge',
   'TautomerEnumerator.Canonicalize'],
  'aggregation_statistic': 'median (per compound, across replicate measurements and pooled activity types)',
  'structural_alert_catalogues': ['PAINS', 'BRENK'],
  'min_assays_filter': 1,
  'rdkit_version': '2026.03.4',
  'bootstrap_resamples_scaffold_ci': 500,
  'nesting_violation_tolerance_pct': 5.0,
  'random_se

In [59]:
{f'{tk}_{pool}': df_save[(tk, pool)].columns.tolist() for tk in TARGETS for pool in POOLS}

{'drd2_ki': ['clean_smiles',
  'molecule_chembl_id',
  'pActivity',
  'median_pooled_concentration_nm',
  'n_measurements',
  'n_assays',
  'n_activity_types',
  'activity_types_used',
  'n_ki',
  'median_pki',
  'median_ki_nm',
  'n_ic50',
  'median_pic50',
  'median_ic50_nm',
  'n_ec50',
  'median_pec50',
  'median_ec50_nm',
  'activity',
  'pActivity_iqr_outlier',
  'is_pains',
  'pains_reason',
  'n_pains_alerts',
  'is_brenk',
  'brenk_reason',
  'n_brenk_alerts',
  'has_any_structural_alert',
  'MW',
  'LogP',
  'TPSA',
  'HBD',
  'HBA',
  'RotBonds',
  'HeavyAtomCount',
  'RingCount',
  'AromaticRingCount',
  'FractionCSP3',
  'Lipinski_compliant',
  'DrugLike_soft',
  'assay_support'],
 'drd2_ki_ic50': ['clean_smiles',
  'molecule_chembl_id',
  'pActivity',
  'median_pooled_concentration_nm',
  'n_measurements',
  'n_assays',
  'n_activity_types',
  'activity_types_used',
  'n_ki',
  'median_pki',
  'median_ki_nm',
  'n_ic50',
  'median_pic50',
  'median_ic50_nm',
  'n_ec50',
 

In [60]:
print('Data cleaning pipeline complete.')
for tk in TARGETS:
    for pool in POOLS:
        print(f'\n{tk.upper()} / {pool}:')
        print(f'  Cleaned records : {len(df_save[(tk, pool)])}')
        print(f'  Output          : data/processed/{output_filename[(tk, pool)]}')

print('\nMaster dataset summary:')
print(dataset_summary_df[['target', 'activity_pool', 'final_unique_compounds', 'active_pct',
                           'n_scaffolds', 'singleton_scaffold_pct', 'pains_free_pct', 'brenk_free_pct']].to_string(index=False))

print(f'\n{len(all_outputs) + len(TARGETS) * len(POOLS)} total output files saved to data/processed/ '
      f'(tables + figures + cleaned datasets), all hashed in the manifest.')


Data cleaning pipeline complete.

DRD2 / ki:
  Cleaned records : 9775
  Output          : data/processed/cleaned_data_drd2_ki.csv

DRD2 / ki_ic50:
  Cleaned records : 10672
  Output          : data/processed/cleaned_data_drd2_ki_ic50.csv

DRD2 / full:
  Cleaned records : 10922
  Output          : data/processed/cleaned_data_drd2_full.csv

CB2 / ki:
  Cleaned records : 5078
  Output          : data/processed/cleaned_data_cb2_ki.csv

CB2 / ki_ic50:
  Cleaned records : 6402
  Output          : data/processed/cleaned_data_cb2_ki_ic50.csv

CB2 / full:
  Cleaned records : 9910
  Output          : data/processed/cleaned_data_cb2_full.csv

ADORA2A / ki:
  Cleaned records : 6563
  Output          : data/processed/cleaned_data_adora2a_ki.csv

ADORA2A / ki_ic50:
  Cleaned records : 8171
  Output          : data/processed/cleaned_data_adora2a_ki_ic50.csv

ADORA2A / full:
  Cleaned records : 8317
  Output          : data/processed/cleaned_data_adora2a_full.csv

OPRM1 / ki:
  Cleaned records : 6289
